# Geothermal Plants, Māori Land Property Titles, Iwi Rohe & 2023 Census (SA2) — Data Processing & Interactive Map Builder (EN & JA)

This notebook processes the GIS and tabular datasets from `plants_data`, `property_data`, `rohe_data`, and the **2023 NZ Census Statistical Area 2 (SA2)** datasets (`2023_census_part_1`, `2023_census_part_2`, `2023_census_work_travel`, `2023_statistical_area_2_clipped`), and builds standalone interactive web map applications in both **English (`map_app/index.html`)** and **Japanese (`map_app/index_ja.html`)** (also synced to `docs/` for GitHub Pages).

### Data Sources Handled
1. **Geothermal Power Stations (`../plants_data/geothermal_plants.gpkg`)**
   - Point locations, operator details, generation technology, and installed capacity (MW) in the Taupō Volcanic Zone.
2. **Māori Land Consolidated Data + Property Titles (`../property_data/20250630_maori_land_consolidated.csv` + `../property_data/nz-property-titles.shp`)**
   - Joins Māori Land Court (MLC) consolidated title records (`LINZ Title Reference`) with LINZ property title boundaries (`title_no`).
   - Distinguishes **Standard Māori Land (`is_aggregated == 0`)** from **Aggregated Ownership (`is_aggregated == 1`, owners shared across multiple blocks)**.
   - Uses `is_rep == 1` to place representative owner-count labels (`No. of Owners`) once per ownership group (`label_group`).
3. **Iwi Areas of Interest / Rohe (`../rohe_data/Iwi_AreasOfInterest.shp`)**
   - Filters Te Puni Kōkiri Iwi Areas of Interest to the Central North Island / Taupō–Reporoa geothermal region.
   - Highlights **Ngāti Tahu / Ngāti Whaoa** and **Ngāti Tūwharetoa** (plus their overlapping area of interest) by default, while keeping neighbouring regional iwi toggleable.
4. **2023 NZ Census SA2 Demographics, Industry & Commuter Flow (`../2023_census_part_1`, `../2023_census_part_2`, `../2023_census_work_travel`, `../2023_statistical_area_2_clipped`)**
   - Processes population counts (usual resident & census night), median age, age structure (`0–14`, `15–64`, `65+`), Māori ethnicity & Māori descent shares, ANZSIC06 19-industry breakdowns (for both **Residents** and **Workplace**), and SA2-to-SA2 commuter flows (internal commuters, external inflow count/rate, and origin SA2s).
   - Interactive map includes a **Regional Report Card & Commuter Inflow Map** (matching the regional socioeconomic report layout with red diagonal hatching for the workplace SA2, sequential blue choropleth for commuter origin SA2s, and dual Top-5 + Other pie charts).

---
### Base Maps Configured
- **Esri World Topo Map** (Default base map layer)
- **Google Maps (Roadmap)**
- **Google Maps (Hybrid Satellite)**
- **Google Maps (Terrain)**

In [1]:
from pathlib import Path
import shutil
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, mapping

# ==============================================================================
# 1. FILE PATHS (Relative to `map_app/` directory)
# ==============================================================================
BASE_DATA_DIR = Path("..")
PLANTS_GPKG_PATH = BASE_DATA_DIR / "plants_data" / "geothermal_plants.gpkg"
# Tip: To use the root-level updated_plants.gpkg instead, uncomment the line below:
# PLANTS_GPKG_PATH = BASE_DATA_DIR / "updated_plants.gpkg"
PLANTS_DESC_EN_DIR = BASE_DATA_DIR / "platns_description_en"
PLANTS_DESC_JP_DIR = BASE_DATA_DIR / "plants_description_jp"

PROPERTY_SHP_PATH = BASE_DATA_DIR / "property_data" / "nz-property-titles.shp"
MAORI_CSV_PATH = BASE_DATA_DIR / "property_data" / "20250630_maori_land_consolidated.csv"
ROHE_SHP_PATH = BASE_DATA_DIR / "rohe_data" / "Iwi_AreasOfInterest.shp"
GEOTHERMAL_SYSTEM_SA_PATH = BASE_DATA_DIR / "geothermal_system_data" / "geothermal_system_large_iwi_SA.geojson"

# 2023 NZ Census & Statistical Area 2 (SA2) paths
SA2_SHP_PATH = BASE_DATA_DIR / "2023_statistical_area_2_clipped" / "statistical-area-2-2023-clipped-generalised.shp"
CENSUS_PART1_CSV_PATH = BASE_DATA_DIR / "2023_census_part_1" / "2023-census-totals-by-topic-for-individuals-by-statistical-a.1.csv"
CENSUS_PART2_CSV_PATH = BASE_DATA_DIR / "2023_census_part_2" / "2023-census-totals-by-topic-for-individuals-by-statistical-a.2.csv"
CENSUS_WORK_TRAVEL_CSV_PATH = BASE_DATA_DIR / "2023_census_work_travel" / "2023-census-main-means-of-travel-to-work-by-statistical-area.csv"

PROCESSED_DIR = Path("processed_data")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_HTML_EN_PATH = Path("index.html")
OUTPUT_HTML_JA_PATH = Path("index_ja.html")
DOCS_DIR = BASE_DATA_DIR / "docs"

# ==============================================================================
# 2. BASE MAP PROVIDERS (Esri World Topo Default + 3 Google Maps Layers)
# ==============================================================================
ESRI_WORLD_TOPO_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}"
GOOGLE_MAPS_ROADMAP_URL = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_MAPS_HYBRID_URL = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"
GOOGLE_MAPS_TERRAIN_URL = "https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}"

# ==============================================================================
# 3. MAP VIEW, OWNER VISIBILITY & GEOMETRY PARAMETERS
# ==============================================================================
DEFAULT_CENTER = [-38.55, 176.18]  # [Latitude, Longitude] centered on Taupō–Reporoa geothermal zone
DEFAULT_ZOOM = 10

# Geometry simplification tolerances (in meters, EPSG:2193) to keep browser rendering fast
PARCEL_SIMPLIFY_TOLERANCE_M = 1.5
ROHE_SIMPLIFY_TOLERANCE_M = 15.0
SA2_SIMPLIFY_TOLERANCE_M = 35.0
COORD_PRECISION = 5      # ~1 meter precision in WGS84 (EPSG:4326) for parcels
SA2_COORD_PRECISION = 4  # ~10 meter precision in WGS84 for regional SA2 boundaries

# Regional bounding boxes (in NZTM EPSG:2193 meters) for SA2 Census extraction
# Core study region: Taupō, Reporoa, Waiotapu, Rotorua, Kawerau, Whakatāne, Matatā-Otakiri, Edgecumbe, Tokoroa, Turangi
SA2_CORE_BBOX_NZTM = (1810000, 5640000, 1975000, 5815000)
# Extended commuter catchment bbox so commuter origin SA2 polygons can be shaded on the map
SA2_CATCHMENT_BBOX_NZTM = (1780000, 5600000, 2010000, 5840000)

# Owner Count Labels visibility on startup (False = hidden by default so map is clean; toggleable in UI)
SHOW_OWNER_LABELS_BY_DEFAULT = False
DEFAULT_MIN_OWNERS_FOR_LABELS = 0  # 0 to max_owners (all)

# ==============================================================================
# 4. IWI ROHE SELECTION & STYLING
# ==============================================================================
# Primary Rohe shown by default on map load
PRIMARY_ROHE_STYLES = {
    "Ngāti Tahu / Ngāti Whaoa": {
        "color": "#be123c",       # Crimson border
        "fillColor": "#f43f5e",   # Rose fill
        "fillOpacity": 0.14,
        "weight": 2.5,
        "dashArray": "6, 4",
        "default_on": True,
    },
    "Ngāti Tūwharetoa": {
        "color": "#0369a1",       # Deep sky-blue border
        "fillColor": "#0ea5e9",   # Sky-blue fill
        "fillOpacity": 0.14,
        "weight": 2.5,
        "dashArray": "8, 4, 2, 4",
        "default_on": True,
    },
}

# Other Central North Island / Geothermal Region Iwi (available as toggleable layers)
SECONDARY_ROHE_PALETTE = [
    ("Ngāti Rangitihi", "#7c3aed"),
    ("Tūhourangi", "#0d9488"),
    ("Raukawa", "#d97706"),
    ("Pouākani (Land Block)", "#4f46e5"),
    ("Tūhoe", "#15803d"),
    ("Ngāti Whakaue", "#b45309"),
    ("Ngāti Mākino", "#9333ea"),
    ("Ngāti Kea / Ngāti Tuarā", "#0891b2"),
    ("Ngāti Rongomai", "#c026d3"),
    ("Ngāti Tarāwhai", "#2563eb"),
    ("Ngāti Rangiwewehi", "#059669"),
    ("Ngāti Pikiao", "#dc2626"),
    ("Ngāti Manawa", "#65a30d"),
    ("Ngāti Whare", "#475569"),
]

# ==============================================================================
# 5. MĀORI LAND & GEOTHERMAL PLANT STYLING
# ==============================================================================
STYLE_STANDARD_MAORI = {
    "color": "#1e293b",       # Slate-800 border
    "fillColor": "#64748b",   # Slate-500 fill (matches report figures)
    "fillOpacity": 0.55,
    "weight": 1.0,
}

STYLE_AGGREGATED_MAORI = {
    "color": "#92400e",       # Amber-800 border
    "fillColor": "#f59e0b",   # Amber-500 / Orange fill (matches report figures)
    "fillOpacity": 0.72,
    "weight": 1.4,
}

PLANT_MARKER_COLOR = "#dc2626"  # Red triangle marker

# ==============================================================================
# 6. GEOTHERMAL SYSTEMS & STATUTORY ACKNOWLEDGEMENT (SA) STYLING
# ==============================================================================
STYLE_GEOSYS_OVERLAP = {
    "color": "#7e22ce",       # Deep purple border (Overlapping SA — multiple settlement groups)
    "fillColor": "#a855f7",   # Purple fill
    "fillOpacity": 0.28,
    "weight": 2.6,
    "dashArray": "",
}

STYLE_GEOSYS_SINGLE = {
    "color": "#0f766e",       # Deep teal border (Single settlement group SA)
    "fillColor": "#14b8a6",   # Teal fill
    "fillOpacity": 0.22,
    "weight": 2.2,
    "dashArray": "",
}

STYLE_GEOSYS_NONE = {
    "color": "#b45309",       # Warm amber-brown dashed border (Non-statutory / Not listed)
    "fillColor": "#f59e0b",   # Amber fill
    "fillOpacity": 0.16,
    "weight": 2.0,
    "dashArray": "6, 4",
}

print("Configuration loaded successfully.")
print("Default base layer: Esri World Topo Map (+ 3 Google Maps layers)")


Configuration loaded successfully.
Default base layer: Esri World Topo Map (+ 3 Google Maps layers)


## Step 1: Load & Process Geothermal Power Stations (`plants_data/geothermal_plants.gpkg`) & Case-Study Description Notes (`platns_description_en/` & `plants_description_jp/`)

We load the 6 geothermal power station locations from `geothermal_plants.gpkg` (in `EPSG:2193` / NZTM2000), convert their coordinates to `EPSG:4326` (WGS84 latitude/longitude), and parse the one-page case-study `.docx` description notes (both English and Japanese editions, including the 5-row specifications table, narrative bullet points, key takeaways, confidence tags, and hyperlinked sources) so that clicking any station icon pops up a full description screen.

In [2]:
import zipfile
import xml.etree.ElementTree as ET

# Load geothermal plants GeoPackage
plants_gdf = gpd.read_file(PLANTS_GPKG_PATH)
print(f"Loaded {len(plants_gdf)} geothermal power stations (CRS: {plants_gdf.crs})")

# Reproject to WGS84 (EPSG:4326) for Leaflet web mapping
plants_wgs84 = plants_gdf.to_crs(epsg=4326).copy()
plants_wgs84["lon"] = plants_wgs84.geometry.x.round(6)
plants_wgs84["lat"] = plants_wgs84.geometry.y.round(6)

# Save processed GeoJSON
plants_geojson_path = PROCESSED_DIR / "geothermal_plants.geojson"
plants_wgs84.to_file(plants_geojson_path, driver="GeoJSON")
print(f"Saved processed plants GeoJSON to: {plants_geojson_path}")

# Parse English and Japanese .docx description files for each of the 6 geothermal power stations
W_NS = "http://schemas.openxmlformats.org/wordprocessingml/wordml"
W_DOC_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
R_DOC_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
DOCX_NS = {"w": W_DOC_NS}

PLANT_DOCX_FILES = {
    "Ohaaki": ("1_Ohaaki.docx", "1_Ohaaki_JP.docx"),
    "Wairakei": ("2_Wairakei.docx", "2_Wairakei_JP.docx"),
    "Mokai": ("3_Mokai.docx", "3_Mokai_JP.docx"),
    "Nga Awa Purua": ("4_Nga_Awa_Purua.docx", "4_Nga_Awa_Purua_JP.docx"),
    "Ngatamariki": ("5_Ngatamariki.docx", "5_Ngatamariki_JP.docx"),
    "Tauhara": ("6_Tauhara.docx", "6_Tauhara_JP.docx"),
}

def parse_plant_docx(docx_path: Path) -> dict:
    """Extract structured title, subtitle, 5-row overview table, narrative sections, and hyperlinked sources from a plant .docx note."""
    if not docx_path.exists():
        return {}
    with zipfile.ZipFile(docx_path) as z:
        rels_root = ET.fromstring(z.read("word/_rels/document.xml.rels"))
        rels_map = {r.attrib["Id"]: r.attrib.get("Target", "") for r in rels_root if "Id" in r.attrib}
        doc_root = ET.fromstring(z.read("word/document.xml"))
        body = doc_root.find("w:body", DOCX_NS)
        paragraphs = []
        specs = []
        sources = []
        in_sources = False
        for child in body:
            tag = child.tag.split("}")[-1]
            if tag == "tbl":
                for row in child.findall(".//w:tr", DOCX_NS):
                    cells = ["".join(t.text or "" for t in tc.findall(".//w:t", DOCX_NS)).strip() for tc in row.findall(".//w:tc", DOCX_NS)]
                    if len(cells) >= 2:
                        specs.append({"label": cells[0], "value": cells[1]})
            elif tag == "p":
                txt = "".join(t.text or "" for t in child.findall(".//w:t", DOCX_NS)).strip()
                if not txt:
                    continue
                if txt.startswith("Sources"):
                    in_sources = True
                    paragraphs.append(txt)
                    continue
                if txt.startswith("Tags:"):
                    in_sources = False
                    paragraphs.append(txt)
                    continue
                if in_sources:
                    hlinks = child.findall(".//w:hyperlink", DOCX_NS)
                    if hlinks:
                        for hl in hlinks:
                            rid = hl.attrib.get(f"{{{R_DOC_NS}}}id", "")
                            htxt = "".join(t.text or "" for t in hl.findall(".//w:t", DOCX_NS)).strip()
                            if htxt:
                                sources.append({"text": htxt, "url": rels_map.get(rid, "")})
                    else:
                        sources.append({"text": txt, "url": ""})
                else:
                    paragraphs.append(txt)
        title = paragraphs[0] if len(paragraphs) > 0 else ""
        subtitle = paragraphs[1] if len(paragraphs) > 1 else ""
        sections = []
        cur_sec = None
        tags_note = ""
        for p in paragraphs[2:]:
            if p.startswith("Tags:"):
                tags_note = p
            elif p.startswith("Social and institutional story") or p.startswith("Where it stands now") or p.startswith("Key takeaway"):
                cur_sec = {"heading": p, "items": []}
                sections.append(cur_sec)
            elif p.startswith("Sources"):
                cur_sec = None
            elif cur_sec is not None:
                cur_sec["items"].append(p)
        return {
            "title": title,
            "subtitle": subtitle,
            "specs": specs,
            "sections": sections,
            "sources": sources,
            "tags_note": tags_note,
        }

plants_list = []
for _, row in plants_wgs84.iterrows():
    pname = str(row["name"])
    en_file, jp_file = PLANT_DOCX_FILES.get(pname, ("", ""))
    desc_en = parse_plant_docx(PLANTS_DESC_EN_DIR / en_file) if en_file else {}
    desc_ja = parse_plant_docx(PLANTS_DESC_JP_DIR / jp_file) if jp_file else {}
    plants_list.append({
        "name": pname,
        "operator": str(row["operator"]),
        "capacity_mw": int(row["capacity_mw"]),
        "type": str(row["type"]),
        "easting": int(row["easting"]),
        "northing": int(row["northing"]),
        "lat": float(row["lat"]),
        "lon": float(row["lon"]),
        "desc_en": desc_en,
        "desc_ja": desc_ja,
    })

with open(PROCESSED_DIR / "plants_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(plants_list, f, ensure_ascii=False, indent=2)
print(f"Parsed English & Japanese .docx description notes for {len(plants_list)} geothermal stations.")

# Display table of geothermal plants
plants_wgs84[["name", "operator", "capacity_mw", "type", "easting", "northing", "lat", "lon"]]


Loaded 6 geothermal power stations (CRS: EPSG:2193)
Saved processed plants GeoJSON to: processed_data\geothermal_plants.geojson
Parsed English & Japanese .docx description notes for 6 geothermal stations.


,name,operator,capacity_mw,type,easting,northing,lat,lon
0,Ohaaki,Contact Energy,44,Flash,1887145,5730689,-38.525896,176.293797
1,Ngatamariki,Mercury NZ,132,Binary,1878510,5728660,-38.546896,176.195700
2,Tauhara,Contact Energy,174,Triple Flash,1874682,5715567,-38.665901,176.157001
3,Nga Awa Purua,Mercury NZ / Tauhara North No.2 Trust,140,Triple Flash,1877209,5721337,-38.613196,176.183703
4,Wairakei,Contact Energy,132,Flash / Binary,1870195,5720210,-38.625500,176.103697
5,Mokai,Tuaropaki Trust / Mercury NZ,113,Combined Cycle (Steam + Binary),1855082,5731196,-38.531100,175.926300


## Step 2: Load & Process Iwi Areas of Interest / Rohe (`rohe_data/Iwi_AreasOfInterest.shp`)

We filter `Iwi_AreasOfInterest.shp` to the Central North Island / Taupō–Reporoa geothermal region iwi configured above. We also compute the exact geometric intersection between **Ngāti Tahu / Ngāti Whaoa** and **Ngāti Tūwharetoa** so the overlapping area of interest can be toggled directly on the map.

In [3]:
# Load Iwi Areas of Interest shapefile (EPSG:2193)
rohe_gdf = gpd.read_file(ROHE_SHP_PATH)
print(f"Total Iwi Rohe in dataset: {len(rohe_gdf)} (CRS: {rohe_gdf.crs})")

# Combine primary and secondary iwi lists
target_iwi_names = list(PRIMARY_ROHE_STYLES.keys()) + [name for name, _ in SECONDARY_ROHE_PALETTE]
rohe_regional = rohe_gdf[rohe_gdf["Name"].isin(target_iwi_names)].copy()

# Calculate area in km2 in NZTM (EPSG:2193) before simplifying
rohe_regional["area_km2"] = (rohe_regional.geometry.area / 1e6).round(1)
rohe_regional["is_primary"] = rohe_regional["Name"].isin(PRIMARY_ROHE_STYLES.keys())

# Compute the overlapping area between Ngāti Tahu / Ngāti Whaoa and Ngāti Tūwharetoa
tahu_geom = rohe_regional.loc[rohe_regional["Name"] == "Ngāti Tahu / Ngāti Whaoa", "geometry"].unary_union
tuwharetoa_geom = rohe_regional.loc[rohe_regional["Name"] == "Ngāti Tūwharetoa", "geometry"].unary_union
overlap_geom_nztm = tahu_geom.intersection(tuwharetoa_geom)
overlap_area_km2 = round(overlap_geom_nztm.area / 1e6, 1)
print(f"Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa overlap area: {overlap_area_km2:,.1f} km²")

# Simplify geometries slightly in NZTM (meters) and reproject to WGS84 (EPSG:4326)
rohe_regional["geometry"] = rohe_regional.geometry.simplify(ROHE_SIMPLIFY_TOLERANCE_M, preserve_topology=True)
rohe_wgs84 = rohe_regional.to_crs(epsg=4326).copy()

overlap_gdf_wgs84 = gpd.GeoDataFrame(
    [{
        "Name": "Overlapping Area (Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa)",
        "Source": "Derived geometric intersection from TPK Iwi Areas of Interest",
        "TPK_Code": "OVL",
        "area_km2": overlap_area_km2,
        "is_primary": True,
    }],
    geometry=[overlap_geom_nztm.simplify(ROHE_SIMPLIFY_TOLERANCE_M, preserve_topology=True)],
    crs=rohe_gdf.crs,
).to_crs(epsg=4326)

# Save processed Rohe GeoJSON
rohe_geojson_path = PROCESSED_DIR / "rohe_central_ni.geojson"
rohe_wgs84.to_file(rohe_geojson_path, driver="GeoJSON")
print(f"Saved {len(rohe_wgs84)} regional Rohe polygons to: {rohe_geojson_path}")

rohe_wgs84[["Name", "TPK_Code", "area_km2", "is_primary", "Source"]].sort_values(
    by=["is_primary", "Name"], ascending=[False, True]
).reset_index(drop=True)


Total Iwi Rohe in dataset: 112 (CRS: EPSG:2193)
Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa overlap area: 1,583.3 km²
Saved 16 regional Rohe polygons to: processed_data\rohe_central_ni.geojson


,Name,TPK_Code,area_km2,is_primary,Source
0,Ngāti Tahu / Ngāti Whaoa,41,1942.1,True,TPK
1,Ngāti Tūwharetoa,42,11335.2,True,TPK
2,Ngāti Kea / Ngāti Tuarā,90,307.9,False,TPK
3,Ngāti Manawa,49,2519.0,False,TPK
4,Ngāti Mākino,30,917.8,False,TPK
5,Ngāti Pikiao,31,1002.1,False,TPK
6,Ngāti Rangitihi,33,3650.4,False,TPK
7,Ngāti Rangiwewehi,34,974.2,False,TPK
8,Ngāti Rongomai,92,44.3,False,TPK
9,Ngāti Tarāwhai,36,25.4,False,TPK


## Step 4: Load & Process 2023 NZ Census SA2 Demographics, Industry & Commuter Flows

In this step, we integrate the four 2023 NZ Census Statistical Area 2 (SA2) datasets:
1. **`statistical-area-2-2023-clipped-generalised.shp`**: 2023 SA2 boundaries (`EPSG:2193`), spatially filtered to our Central North Island / Bay of Plenty study region (`SA2_CORE_BBOX_NZTM`) plus any regional commuter origin SA2s (`SA2_CATCHMENT_BBOX_NZTM`).
2. **`2023-census-totals-by-topic-for-individuals-by-statistical-a.1.csv` (Part 1)**:
   - Usual resident population (`VAR_1_3`), Census night population (`VAR_1_6`), Median age (`VAR_1_69`).
   - Age distribution: Youth 0–14 yrs (`VAR_1_80`), Working Age 15–64 yrs (`VAR_1_81 + VAR_1_82`), Seniors 65+ yrs (`VAR_1_83`), Total (`VAR_1_84`).
   - Māori ethnicity (`VAR_1_159 / VAR_1_168`) and Māori descent (`VAR_1_181 / VAR_1_186`).
3. **`2023-census-totals-by-topic-for-individuals-by-statistical-a.2.csv` (Part 2)**:
   - 19 ANZSIC06 industry categories for **Usual Residents (`VAR_2_591`–`VAR_2_609`, Total `VAR_2_611`)** and **Workplace Employed (`VAR_2_657`–`VAR_2_675`, Total `VAR_2_677`)**.
   - Computes **Top 5 Industries + Other** percentages for both Resident and Workplace pie charts (dividing by the sum of positive industry counts to account for Stats NZ base-3 random rounding).
4. **`2023-census-main-means-of-travel-to-work-by-statistical-area.csv` (Work Travel)**:
   - Aggregates SA2-to-SA2 commuter flows (`2023_Total_stated`) by `SA22023_V1_00_workplace_address` to compute total commuter flow base, internal commuters (`residence == workplace`), external commuter inflow count & rate (%), and the ranked list of inflow origin SA2s.

In [5]:
def round_coords(coords, precision=COORD_PRECISION):
    """Recursively round GeoJSON coordinate arrays to keep the standalone HTML compact and fast."""
    if isinstance(coords, (list, tuple)):
        if len(coords) > 0 and isinstance(coords[0], (int, float)):
            return [round(float(c), precision) for c in coords]
        return [round_coords(c, precision) for c in coords]
    return coords

def clean_val(val):
    """Convert Stats NZ suppressed/confidential negative codes (-999, -998, -997) or NaNs to 0."""
    if pd.isna(val):
        return 0.0
    try:
        v = float(val)
        return 0.0 if v < 0 else v
    except Exception:
        return 0.0

# 1. Load SA2 2023 clipped boundaries (EPSG:2193)
sa2_gdf = gpd.read_file(SA2_SHP_PATH)
sa2_gdf["SA22023_V1"] = sa2_gdf["SA22023_V1"].astype(str).str.strip()

# Filter to core regional bounding box & broader commuter catchment bounding box
cx0, cy0, cx1, cy1 = SA2_CORE_BBOX_NZTM
ex0, ey0, ex1, ey1 = SA2_CATCHMENT_BBOX_NZTM
core_sa2_gdf = sa2_gdf.cx[cx0:cx1, cy0:cy1].copy()
catchment_sa2_gdf = sa2_gdf.cx[ex0:ex1, ey0:ey1].copy()

# Exclude oceanic/inlet water polygons with 0 land area if needed, but keep all land SA2s
core_sa2_codes = set(core_sa2_gdf["SA22023_V1"].unique())
catchment_sa2_codes = set(catchment_sa2_gdf["SA22023_V1"].unique())
print(f"Core study region SA2 count: {len(core_sa2_codes)} | Catchment SA2 count: {len(catchment_sa2_codes)}")

# 2. Load Work Travel (SA2-to-SA2 commuter matrix)
wt_df = pd.read_csv(CENSUS_WORK_TRAVEL_CSV_PATH)
wt_df["res_code"] = wt_df["SA22023_V1_00_usual_residence_address"].astype(str).str.strip()
wt_df["work_code"] = wt_df["SA22023_V1_00_workplace_address"].astype(str).str.strip()
wt_df["res_name"] = wt_df["SA22023_V1_00_NAME_usual_residence_address"].astype(str).str.strip()
wt_df["work_name"] = wt_df["SA22023_V1_00_NAME_workplace_address"].astype(str).str.strip()
wt_df["count_2023"] = wt_df["2023_Total_stated"].apply(lambda v: int(clean_val(v)))

# Identify all commuter origin SA2s that send workers into our core study region SA2s
wt_core = wt_df[(wt_df["work_code"].isin(core_sa2_codes)) & (wt_df["count_2023"] > 0)].copy()
origin_codes_in_catchment = set(wt_core["res_code"].unique()).intersection(catchment_sa2_codes)
all_selected_sa2_codes = sorted(core_sa2_codes.union(origin_codes_in_catchment))
print(f"Total SA2 polygons selected (Core + Regional Inflow Origins): {len(all_selected_sa2_codes)}")

# Build commuter inflow dictionary keyed by workplace SA2 code
commute_by_workplace = {}
for work_code, grp in wt_df[(wt_df["work_code"].isin(all_selected_sa2_codes)) & (wt_df["count_2023"] > 0)].groupby("work_code"):
    total_base = int(grp["count_2023"].sum())
    internal_cnt = int(grp.loc[grp["res_code"] == work_code, "count_2023"].sum())
    inflow_cnt = total_base - internal_cnt
    inflow_pct = round(inflow_cnt / total_base * 100.0, 1) if total_base > 0 else 0.0
    ext_rows = grp[grp["res_code"] != work_code].sort_values(by="count_2023", ascending=False)
    origins_list = [
        {"code": str(r["res_code"]), "name": str(r["res_name"]), "count": int(r["count_2023"])}
        for _, r in ext_rows.iterrows()
    ]
    commute_by_workplace[work_code] = {
        "base": total_base,
        "internal": internal_cnt,
        "inflow": inflow_cnt,
        "inflow_pct": inflow_pct,
        "origins": origins_list,
    }

# 3. Load 2023 Census Part 1 (Demographics, Age, Ethnicity, Māori Descent) — skip huge WKT column
c1_df = pd.read_csv(CENSUS_PART1_CSV_PATH, usecols=lambda c: c != "WKT", low_memory=False)
c1_df["sa2_code"] = c1_df["SA22023_V1_00"].astype(str).str.strip()
c1_sub = c1_df[c1_df["sa2_code"].isin(all_selected_sa2_codes)].set_index("sa2_code")

# 4. Load 2023 Census Part 2 (19 ANZSIC06 Industries: Residents vs. Workplace) — skip huge WKT column
c2_df = pd.read_csv(CENSUS_PART2_CSV_PATH, usecols=lambda c: c != "WKT", low_memory=False)
c2_df["sa2_code"] = c2_df["SA22023_V1_00"].astype(str).str.strip()
c2_sub = c2_df[c2_df["sa2_code"].isin(all_selected_sa2_codes)].set_index("sa2_code")

# 19 ANZSIC06 Industry definitions: (Resident Col, Workplace Col, English Label, Japanese Label)
INDUSTRY_SPECS = [
    ("VAR_2_591", "VAR_2_657", "Agriculture, Forestry & Fishing", "農林水産業"),
    ("VAR_2_592", "VAR_2_658", "Mining", "鉱業"),
    ("VAR_2_593", "VAR_2_659", "Manufacturing", "製造業"),
    ("VAR_2_594", "VAR_2_660", "Electricity, Gas, Water & Waste", "インフラ・廃棄物処理業"),
    ("VAR_2_595", "VAR_2_661", "Construction", "建設業"),
    ("VAR_2_596", "VAR_2_662", "Wholesale Trade", "卸売業"),
    ("VAR_2_597", "VAR_2_663", "Retail Trade", "小売業"),
    ("VAR_2_598", "VAR_2_664", "Accommodation & Food Services", "宿泊・飲食サービス業"),
    ("VAR_2_599", "VAR_2_665", "Transport, Postal & Warehousing", "運輸・物流・倉庫業"),
    ("VAR_2_600", "VAR_2_666", "Information Media & Telecom", "情報通信・メディア業"),
    ("VAR_2_601", "VAR_2_667", "Financial & Insurance Services", "金融・保険業"),
    ("VAR_2_602", "VAR_2_668", "Rental, Hiring & Real Estate", "不動産・物品賃貸業"),
    ("VAR_2_603", "VAR_2_669", "Professional, Scientific & Tech", "学術・専門・技術サービス業"),
    ("VAR_2_604", "VAR_2_670", "Administrative & Support Services", "行政支援・人材サービス業"),
    ("VAR_2_605", "VAR_2_671", "Public Administration & Safety", "公務・防衛・安全業"),
    ("VAR_2_606", "VAR_2_672", "Education & Training", "教育・訓練業"),
    ("VAR_2_607", "VAR_2_673", "Health Care & Social Assistance", "医療・社会福祉業"),
    ("VAR_2_608", "VAR_2_674", "Arts & Recreation Services", "芸術・娯楽・レジャー業"),
    ("VAR_2_609", "VAR_2_675", "Other Services", "その他サービス業"),
]

def compute_top5_industries(row_c2, is_workplace=False):
    """
    Compute Top 5 + Other industry breakdown using single-stage direct calculation:
      - Uses the raw Stats NZ base-3 random-rounded headcounts for all 19 ANZSIC06 categories.
      - Divides each category count directly by the sum of the 19 industry counts (`total_pos`)
        and rounds once to 1 decimal place (`round(count / total_pos * 100.0, 1)`).
    """
    items = []
    for res_col, work_col, en_name, ja_name in INDUSTRY_SPECS:
        col = work_col if is_workplace else res_col
        cnt = int(clean_val(row_c2.get(col, 0))) if row_c2 is not None else 0
        if cnt > 0:
            items.append({"en": en_name, "ja": ja_name, "count": cnt})
    total_pos = sum(x["count"] for x in items)
    if total_pos <= 0:
        return [], 0
    items.sort(key=lambda x: -x["count"])
    top5 = items[:5]
    other_cnt = sum(x["count"] for x in items[5:])
    result = []
    for x in top5:
        result.append({
            "en": x["en"],
            "ja": x["ja"],
            "count": x["count"],
            "pct": round(x["count"] / total_pos * 100.0, 1),
        })
    if other_cnt > 0:
        result.append({
            "en": "Other",
            "ja": "その他",
            "count": other_cnt,
            "pct": round(other_cnt / total_pos * 100.0, 1),
            "is_other": True,
        })
    return result, total_pos

# 5. Spatial join to check which SA2 polygons contain Geothermal Power Stations
sa2_selected_gdf = sa2_gdf[sa2_gdf["SA22023_V1"].isin(all_selected_sa2_codes)].copy()
plants_in_sa2 = gpd.sjoin(plants_gdf[["name", "capacity_mw", "geometry"]], sa2_selected_gdf[["SA22023_V1", "geometry"]], how="inner", predicate="within")
plants_by_sa2 = {}
for _, prow in plants_in_sa2.iterrows():
    scode = str(prow["SA22023_V1"])
    plants_by_sa2.setdefault(scode, []).append(f"{prow['name']} ({prow['capacity_mw']} MW)")

# Compute representative label points in WGS84 and simplify SA2 geometries for fast browser rendering
sa2_rep_pts_wgs84 = gpd.GeoSeries(sa2_selected_gdf.geometry.representative_point(), crs=sa2_selected_gdf.crs).to_crs(epsg=4326)
sa2_selected_gdf["rep_lat"] = sa2_rep_pts_wgs84.y.round(4)
sa2_selected_gdf["rep_lon"] = sa2_rep_pts_wgs84.x.round(4)
sa2_selected_gdf["geometry"] = sa2_selected_gdf.geometry.simplify(SA2_SIMPLIFY_TOLERANCE_M, preserve_topology=True)
sa2_selected_wgs84 = sa2_selected_gdf.to_crs(epsg=4326)

# 6. Assemble enriched SA2 features and summary table
census_sa2_features = []
summary_rows = []

for _, srow in sa2_selected_wgs84.iterrows():
    scode = str(srow["SA22023_V1"])
    sname = str(srow.get("SA22023__1") or srow.get("SA22023_1") or scode)
    area_km2 = round(float(srow.get("LAND_AREA_", 0.0)), 2)
    is_core = scode in core_sa2_codes

    r1 = c1_sub.loc[scode] if scode in c1_sub.index else None
    r2 = c2_sub.loc[scode] if scode in c2_sub.index else None

    if r1 is not None and pd.notna(r1.get("SA22023_V1_00_NAME")):
        sname = str(r1["SA22023_V1_00_NAME"]).strip()

    pop_usual = int(clean_val(r1["VAR_1_3"])) if r1 is not None else 0
    pop_night = int(clean_val(r1["VAR_1_6"])) if r1 is not None else 0
    median_age = round(float(clean_val(r1["VAR_1_69"])), 1) if r1 is not None else 0.0

    age_0_14 = int(clean_val(r1["VAR_1_80"])) if r1 is not None else 0
    age_15_64 = int(clean_val(r1["VAR_1_81"]) + clean_val(r1["VAR_1_82"])) if r1 is not None else 0
    age_65_plus = int(clean_val(r1["VAR_1_83"])) if r1 is not None else 0
    age_tot = int(clean_val(r1["VAR_1_84"])) if r1 is not None else 0
    if age_tot <= 0:
        age_tot = age_0_14 + age_15_64 + age_65_plus

    age_0_14_pct = round(age_0_14 / age_tot * 100.0, 1) if age_tot > 0 else 0.0
    age_15_64_pct = round(age_15_64 / age_tot * 100.0, 1) if age_tot > 0 else 0.0
    age_65_plus_pct = round(age_65_plus / age_tot * 100.0, 1) if age_tot > 0 else 0.0

    eth_maori = int(clean_val(r1["VAR_1_159"])) if r1 is not None else 0
    eth_tot = int(clean_val(r1["VAR_1_168"])) if r1 is not None else 0
    maori_eth_pct = round(eth_maori / eth_tot * 100.0, 1) if eth_tot > 0 else 0.0

    desc_maori = int(clean_val(r1["VAR_1_181"])) if r1 is not None else 0
    desc_tot = int(clean_val(r1["VAR_1_186"])) if r1 is not None else 0
    maori_desc_pct = round(desc_maori / desc_tot * 100.0, 1) if desc_tot > 0 else 0.0

    res_emp_total = int(clean_val(r2["VAR_2_611"])) if r2 is not None else 0
    work_emp_total = int(clean_val(r2["VAR_2_677"])) if r2 is not None else 0

    res_top5, res_ind_sum = compute_top5_industries(r2, is_workplace=False)
    work_top5, work_ind_sum = compute_top5_industries(r2, is_workplace=True)

    # Specifically track Electricity/Gas/Water/Waste + Mining (Geothermal-related utilities & extractive sector)
    work_egww_mining = int(clean_val(r2["VAR_2_658"]) + clean_val(r2["VAR_2_660"])) if r2 is not None else 0
    work_egww_mining_pct = round(work_egww_mining / work_ind_sum * 100.0, 1) if work_ind_sum > 0 else 0.0

    comm_info = commute_by_workplace.get(scode, {
        "base": 0, "internal": 0, "inflow": 0, "inflow_pct": 0.0, "origins": []
    })

    geom_dict = mapping(srow.geometry)
    geom_dict["coordinates"] = round_coords(geom_dict["coordinates"], SA2_COORD_PRECISION)

    props = {
        "sa2_code": scode,
        "sa2_name": sname,
        "is_core": bool(is_core),
        "area_km2": area_km2,
        "rep_lat": float(srow["rep_lat"]),
        "rep_lon": float(srow["rep_lon"]),
        "pop_usual": pop_usual,
        "pop_night": pop_night,
        "median_age": median_age,
        "age_0_14_pct": age_0_14_pct,
        "age_15_64_pct": age_15_64_pct,
        "age_65_plus_pct": age_65_plus_pct,
        "maori_eth_pct": maori_eth_pct,
        "maori_desc_pct": maori_desc_pct,
        "res_emp_total": res_emp_total,
        "work_emp_total": work_emp_total,
        "work_egww_mining": work_egww_mining,
        "work_egww_mining_pct": work_egww_mining_pct,
        "commute_base": comm_info["base"],
        "commute_internal": comm_info["internal"],
        "commute_inflow": comm_info["inflow"],
        "commute_inflow_pct": comm_info["inflow_pct"],
        "inflow_origins": comm_info["origins"],
        "res_top5": res_top5,
        "work_top5": work_top5,
        "plants": plants_by_sa2.get(scode, []),
    }
    census_sa2_features.append({
        "type": "Feature",
        "geometry": geom_dict,
        "properties": props,
    })

    if is_core:
        summary_rows.append({
            "sa2_code": scode,
            "sa2_name": sname,
            "pop_usual": pop_usual,
            "pop_night": pop_night,
            "median_age": median_age,
            "maori_eth_pct": maori_eth_pct,
            "maori_desc_pct": maori_desc_pct,
            "work_emp_total": work_emp_total,
            "commute_base": comm_info["base"],
            "commute_internal": comm_info["internal"],
            "commute_inflow": comm_info["inflow"],
            "commute_inflow_pct": comm_info["inflow_pct"],
            "top_work_industry": work_top5[0]["en"] if work_top5 else "-",
            "geothermal_plants": ", ".join(plants_by_sa2.get(scode, [])),
        })

census_sa2_geojson_obj = {"type": "FeatureCollection", "features": census_sa2_features}

# Save processed Census SA2 GeoJSON and CSV summary
census_geojson_path = PROCESSED_DIR / "census_sa2_2023.geojson"
with open(census_geojson_path, "w", encoding="utf-8") as f:
    json.dump(census_sa2_geojson_obj, f, ensure_ascii=False)

census_summary_df = pd.DataFrame(summary_rows).sort_values(by="sa2_name").reset_index(drop=True)
census_csv_path = PROCESSED_DIR / "census_sa2_summary.csv"
census_summary_df.to_csv(census_csv_path, index=False, encoding="utf-8-sig")
print(f"Saved processed 2023 Census SA2 GeoJSON ({len(census_sa2_features)} features) to: {census_geojson_path}")
print(f"Saved core regional SA2 Census summary CSV ({len(census_summary_df)} rows) to: {census_csv_path}")

# Display verification table for key geothermal & report SA2 areas
key_highlight_codes = ["188200", "187800", "187701", "201800", "201600", "202000", "202900", "188900"]
census_summary_df[census_summary_df["sa2_code"].isin(key_highlight_codes)].reset_index(drop=True)


Core study region SA2 count: 173 | Catchment SA2 count: 446
Total SA2 polygons selected (Core + Regional Inflow Origins): 359
Saved processed 2023 Census SA2 GeoJSON (359 features) to: processed_data\census_sa2_2023.geojson
Saved core regional SA2 Census summary CSV (173 rows) to: processed_data\census_sa2_summary.csv


,sa2_code,sa2_name,pop_usual,pop_night,median_age,maori_eth_pct,maori_desc_pct,work_emp_total,commute_base,commute_internal,commute_inflow,commute_inflow_pct,top_work_industry,geothermal_plants
0,202900,Allandale,2613,2595,42.5,39.7,41.8,1110,1032,255,777,75.3,Education & Training,
1,201800,Golden Springs,1854,1848,33.0,31.1,34.1,1074,921,612,309,33.6,"Agriculture, Forestry & Fishing",
2,187701,Marotiri,1581,1557,34.6,29.6,32.4,867,774,540,234,30.2,"Agriculture, Forestry & Fishing",Mokai (113 MW)
3,202000,Matatā-Otakiri,1746,1719,40.8,39.2,43.1,438,384,327,57,14.8,"Agriculture, Forestry & Fishing",
4,188900,Mountview,2838,2919,33.9,37.6,40.9,627,591,240,351,59.4,Education & Training,
5,201600,Ngakuru,1878,1914,36.9,22.2,26.2,645,537,513,24,4.5,"Agriculture, Forestry & Fishing",
6,187800,Ohakuri,1953,1944,38.7,22.1,25.7,669,570,474,96,16.8,"Agriculture, Forestry & Fishing","Ohaaki (44 MW), Ngatamariki (132 MW)"
7,188200,Wairākei-Broadlands,1182,1203,35.1,36.5,38.6,1035,897,219,678,75.6,"Electricity, Gas, Water & Waste","Tauhara (174 MW), Nga Awa Purua (140 MW), Wair..."


## Step 6: Build Dedicated 2023 Census & Commuter Flow Report Apps (`census.html` & `census_ja.html`)

To keep the user interface clean, spacious, and easy to read, we build a **separate, dedicated 2023 Census & Industry / Commuter Flow web application**:
- **`census.html`**: English edition (with header navigation back to `index.html` and a `[ EN | JA ]` language switch).
- **`census_ja.html`**: Japanese edition (with header navigation back to `index_ja.html` and a `[ EN | JA ]` language switch).

### Layout & Features (Matching the Regional Socioeconomic Report)
- **Left Panel (50% Width — Dedicated Regional Report Sheet)**:
  - Large SA2 header (`<SA2 Name> (<SA2 Code>)`), e.g. `Matatā-Otakiri (202000)` or `Wairākei-Broadlands (188200)`.
  - Spacious 5-line demographic & commuter flow summary box (`Usual Resident Pop / Night Pop / Median Age`, `Age Structure`, `Māori Ethnicity / Descent`, `Workplace Employed / Commuter Flow Base`, and clickable `Inflow Origins`).
  - Two large side-by-side SVG Pie Charts (`[Residents] Employment by Industry (Top 5 + Other)` and `[Workplace] Area Industry Composition (Top 5 + Other)`) with full, untruncated industry tables below each chart.
- **Right Panel (50% Width — Dedicated Commuter Inflow & SA2 Choropleth Map)**:
  - Highlights the selected workplace SA2 with **red diagonal hatching (`就業地`)** and shades all commuter inflow origin SA2 polygons on a **sequential blue choropleth scale (`通勤者数 (人)`)**, automatically framing the commuter catchment extent.

In [7]:
OUTPUT_CENSUS_EN_PATH = Path("census.html")
OUTPUT_CENSUS_JA_PATH = Path("census_ja.html")

CENSUS_UI_TEXT = {
    "en": {
        "html_lang": "en",
        "page_title": "2023 NZ Census Regional Socioeconomic & Commuter Flow Report (SA2)",
        "app_title": "2023 NZ Census — Regional Socioeconomic & Commuter Explorer",
        "app_sub": "Taupō, Reporoa, Rotorua & Bay of Plenty Statistical Area 2 (SA2) Demographics, Industry & Commuter Flows",
        "nav_land_map": "🏞️ Māori Land & Rohe Map",
        "nav_census_map": "📊 2023 Census Report App",
        "land_href": "index.html",
        "select_sa2_label": "Target SA2 Area:",
        "search_placeholder": "Filter SA2 by name or code (e.g. Matatā, Wairākei, 202000)...",
        "quick_areas_label": "Key Areas:",
        "map_mode_label": "Map Layer Mode:",
        "cm_commute": "Commuter Inflow Map (Report Style)",
        "cm_pop": "Usual Resident Population (2023)",
        "cm_maori": "Māori Ethnicity Share (%)",
        "cm_work_emp": "Workplace Employed Total",
        "cm_geothermal_ind": "Utility & Mining Share (%)",
        "basemap_label": "Base Map:",
        "bm_esri_topo": "Esri World Topo Map (Default)",
        "bm_google_road": "Google Maps (Roadmap)",
        "bm_google_hybrid": "Google Maps (Hybrid Satellite)",
        "bm_google_terrain": "Google Maps (Terrain)",
        "chk_sa2_labels": "SA2 Labels",
        "chk_plants": "Geothermal Plants (6)",
        "chk_rohe": "Primary Iwi Rohe",
        "btn_fit_flow": "🎯 Fit Commuter Catchment",
        "btn_zoom_sa2": "🔍 Zoom to SA2",
        "pie_res_title": "[Residents] Employment by Industry (Top 5 + Other)",
        "pie_work_title": "[Workplace] Area Industry Composition (Top 5 + Other)",
        "tbl_col_industry": "Industry Category (ANZSIC06)",
        "tbl_col_count": "Workers",
        "tbl_col_pct": "Share",
        "no_ind_data": "Confidential / Suppressed or no industry data recorded",
        "none_inflow": "None recorded (or suppressed)",
        "workplace_prefix": "Workplace",
        "inflow_origin_label": "Inflow Origin SA2",
        "cb_commuters": "Commuters (persons)",
        "cb_pop": "Usual Residents",
        "cb_maori": "Māori Ethnicity (%)",
        "cb_work_emp": "Workplace Employed",
        "cb_geo_ind": "Utility & Mining (%)",
        "click_map_hint": "💡 Tip: Click any SA2 polygon on the map or any Inflow Origin tag to switch the report to that area. (Note: Industry shares are calculated directly from Stats NZ base-3 random-rounded headcounts and rounded to 1 decimal place; totals may not equal 100.0% due to rounding.)",
        "plant_badge_prefix": "⚡ Geothermal Station in SA2:",
    },
    "ja": {
        "html_lang": "ja",
        "page_title": "2023年NZ国勢調査 地域別レポート（人口・産業構造・通勤流動）",
        "app_title": "2023年NZ国勢調査 — 地域別レポート（人口・産業構造・通勤流動）",
        "app_sub": "タウポ・レポロア・ロトルア・ベイオブプレンティ周辺 統計区域2 (SA2) 人口動態・就業産業・通勤流入元マップ",
        "nav_land_map": "🏞️ マオリ土地・ロヘマップへ",
        "nav_census_map": "📊 2023年国勢調査レポート",
        "land_href": "index_ja.html",
        "select_sa2_label": "対象SA2エリア:",
        "search_placeholder": "地域名・SA2コードで検索 (例: Matatā, Wairākei, 202000)...",
        "quick_areas_label": "主要エリア:",
        "map_mode_label": "地図表示モード:",
        "cm_commute": "通勤流動マップ（レポート形式：就業地＋流入元）",
        "cm_pop": "通常居住人口 (2023年)",
        "cm_maori": "Māori 民族割合 (%)",
        "cm_work_emp": "就業者総数（就業地ベース）",
        "cm_geothermal_ind": "インフラ・廃棄物処理業＋鉱業 比率 (%)",
        "basemap_label": "背景地図:",
        "bm_esri_topo": "Esri World Topo Map（標準地形図・デフォルト）",
        "bm_google_road": "Google マップ（道路地図）",
        "bm_google_hybrid": "Google マップ（衛星ハイブリッド）",
        "bm_google_terrain": "Google マップ（地形図 Terrain）",
        "chk_sa2_labels": "地域名・人数ラベル表示",
        "chk_plants": "地熱発電所 (6箇所)",
        "chk_rohe": "主要イウィ・ロヘ境界",
        "btn_fit_flow": "🎯 通勤圏全体を表示",
        "btn_zoom_sa2": "🔍 対象エリアを拡大",
        "pie_res_title": "【住民】就業産業構成 (Top 5 + その他)",
        "pie_work_title": "【地域産業】エリア内産業構成 (Top 5 + その他)",
        "tbl_col_industry": "産業分類 (ANZSIC06)",
        "tbl_col_count": "就業者数",
        "tbl_col_pct": "構成比",
        "no_ind_data": "秘匿（Confidential）またはデータなし",
        "none_inflow": "該当なし（または秘匿）",
        "workplace_prefix": "就業地",
        "inflow_origin_label": "流入元地域",
        "cb_commuters": "通勤者数 (人)",
        "cb_pop": "通常居住人口 (人)",
        "cb_maori": "Māori 民族割合 (%)",
        "cb_work_emp": "就業者総数 (人)",
        "cb_geo_ind": "インフラ・鉱業 (%)",
        "click_map_hint": "💡 ヒント: 右側の地図上のSA2ポリゴン、またはレポート内の「流入元地域」タグをクリックすると対象地域を切り替えられます。（※注: 産業構成比はStats NZの3進法ランダム丸め後の就業者数から直接算出し小数第1位に四捨五入しているため、合計が100.0%にならない場合があります。）",
        "plant_badge_prefix": "⚡ エリア内の地熱発電所:",
    },
}

def build_census_html_app(lang_code: str) -> str:
    ct = CENSUS_UI_TEXT[lang_code]
    active_en = "active" if lang_code == "en" else ""
    active_ja = "active" if lang_code == "ja" else ""
    primary_rohe_only = [r for r in rohe_layers_data if r["category"] in ("primary", "overlap")]
    return f"""<!DOCTYPE html>
<html lang="{ct['html_lang']}">
<head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>{ct['page_title']}</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        :root {{
            --bg-main: #f1f5f9;
            --bg-card: #ffffff;
            --border-color: #cbd5e1;
            --text-main: #0f172a;
            --text-muted: #475569;
            --accent: #2563eb;
            --accent-sky: #0284c7;
            --workplace-red: #dc2626;
        }}
        *, *::before, *::after {{
            box-sizing: border-box;
        }}
        html, body {{
            margin: 0;
            padding: 0;
            width: 100%;
            height: 100%;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Hiragino Sans", "Yu Gothic UI", "Meiryo", Roboto, sans-serif;
            color: var(--text-main);
            background: var(--bg-main);
            overflow: hidden;
            -webkit-text-size-adjust: 100%;
        }}
        #app-wrapper {{
            display: flex;
            flex-direction: column;
            width: 100%;
            max-width: 100vw;
            height: 100%;
            overflow: hidden;
        }}
        /* Top Header Bar */
        .top-header {{
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
            color: #ffffff;
            padding: 10px 18px;
            display: flex;
            justify-content: space-between;
            align-items: center;
            gap: 14px;
            border-bottom: 1px solid #334155;
            flex-shrink: 0;
            z-index: 1010;
        }}
        .header-brand h1 {{
            margin: 0;
            font-size: 16px;
            font-weight: 700;
            letter-spacing: 0.2px;
        }}
        .header-brand p {{
            margin: 2px 0 0 0;
            font-size: 11.5px;
            color: #cbd5e1;
        }}
        .header-nav-right {{
            display: flex;
            align-items: center;
            gap: 10px;
            flex-shrink: 0;
        }}
        .app-nav-pills {{
            display: flex;
            gap: 6px;
        }}
        .app-nav-btn {{
            padding: 6px 12px;
            border-radius: 6px;
            font-size: 12px;
            font-weight: 700;
            text-decoration: none;
            color: rgba(255, 255, 255, 0.85);
            background: rgba(255, 255, 255, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.24);
            transition: all 0.15s ease;
            white-space: nowrap;
        }}
        .app-nav-btn:hover {{
            background: rgba(255, 255, 255, 0.22);
            color: #ffffff;
        }}
        .app-nav-btn.active {{
            background: #0284c7;
            color: #ffffff;
            border-color: #38bdf8;
            pointer-events: none;
        }}
        .lang-toggle {{
            display: inline-flex;
            align-items: center;
            background: rgba(15, 23, 42, 0.55);
            border: 1px solid rgba(255, 255, 255, 0.3);
            border-radius: 20px;
            padding: 2px;
            gap: 2px;
        }}
        .lang-toggle-opt {{
            padding: 4px 11px;
            font-size: 11.5px;
            font-weight: 700;
            text-decoration: none;
            border-radius: 16px;
            color: rgba(255, 255, 255, 0.75);
            transition: all 0.15s;
        }}
        .lang-toggle-opt:hover {{
            color: #ffffff;
            background: rgba(255, 255, 255, 0.15);
        }}
        .lang-toggle-opt.active {{
            background: #ffffff;
            color: #0f172a;
            pointer-events: none;
        }}
        /* Sub-header Quick Area Toolbar */
        .quick-toolbar {{
            background: #ffffff;
            border-bottom: 1px solid var(--border-color);
            padding: 8px 18px;
            display: flex;
            flex-wrap: wrap;
            align-items: center;
            gap: 10px;
            flex-shrink: 0;
            box-shadow: 0 1px 4px rgba(15, 23, 42, 0.04);
        }}
        .toolbar-group {{
            display: flex;
            align-items: center;
            gap: 6px;
        }}
        .toolbar-label {{
            font-size: 12px;
            font-weight: 700;
            color: #334155;
            white-space: nowrap;
        }}
        .select-ctrl {{
            padding: 5px 10px;
            font-size: 13px;
            font-weight: 600;
            border: 1px solid #94a3b8;
            border-radius: 6px;
            background: #ffffff;
            color: #0f172a;
            cursor: pointer;
        }}
        .search-ctrl {{
            padding: 5px 10px;
            font-size: 12.5px;
            border: 1px solid #cbd5e1;
            border-radius: 6px;
            width: 240px;
        }}
        .chip-row {{
            display: flex;
            flex-wrap: wrap;
            gap: 5px;
            align-items: center;
        }}
        .area-chip {{
            background: #f8fafc;
            border: 1px solid #cbd5e1;
            border-radius: 6px;
            padding: 4px 10px;
            font-size: 12px;
            font-weight: 600;
            color: #1e293b;
            cursor: pointer;
            transition: all 0.15s;
        }}
        .area-chip:hover {{
            background: #e0f2fe;
            border-color: #38bdf8;
        }}
        .area-chip.active {{
            background: #2563eb;
            color: #ffffff;
            border-color: #1d4ed8;
        }}
        /* Main Split Content (50% Left Report Sheet / 50% Right Map) */
        #main-split {{
            flex: 1;
            display: grid;
            grid-template-columns: 52% 48%;
            width: 100%;
            min-height: 0;
        }}
        /* Left Panel: Dedicated Regional Report Sheet */
        #report-pane {{
            background: #f8fafc;
            border-right: 1px solid var(--border-color);
            overflow-y: auto;
            padding: 16px 20px 28px 20px;
        }}
        .report-sheet {{
            background: #ffffff;
            border: 1px solid #cbd5e1;
            border-radius: 10px;
            padding: 18px 22px 22px 22px;
            box-shadow: 0 2px 10px rgba(15, 23, 42, 0.05);
        }}
        .report-title-row {{
            display: flex;
            justify-content: space-between;
            align-items: flex-end;
            border-bottom: 3px solid #2563eb;
            padding-bottom: 8px;
            margin-bottom: 14px;
            gap: 12px;
            flex-wrap: wrap;
        }}
        .report-title-row h2 {{
            margin: 0;
            font-size: 22px;
            font-weight: 800;
            color: #1e293b;
            letter-spacing: 0.2px;
        }}
        .report-title-actions {{
            display: flex;
            gap: 6px;
        }}
        .btn-action {{
            background: #f1f5f9;
            border: 1px solid #cbd5e1;
            border-radius: 6px;
            padding: 5px 10px;
            font-size: 12px;
            font-weight: 600;
            color: #1e293b;
            cursor: pointer;
            transition: all 0.15s;
        }}
        .btn-action:hover {{
            background: #e2e8f0;
        }}
        .plant-alert-banner {{
            background: #fef2f2;
            border: 1px solid #fecaca;
            color: #991b1b;
            border-radius: 6px;
            padding: 6px 12px;
            font-size: 12.5px;
            font-weight: 600;
            margin-bottom: 12px;
            display: none;
        }}
        /* Visual Demographic & Commuter Summary Dashboard */
        .summary-box {{
            background: #f8fafc;
            border: 1.5px solid #cbd5e1;
            border-radius: 10px;
            padding: 12px 14px;
            color: #0f172a;
            margin-bottom: 16px;
            display: flex;
            flex-direction: column;
            gap: 10px;
            width: 100%;
            max-width: 100%;
            box-sizing: border-box;
            overflow: hidden;
        }}
        .kpi-strip {{
            display: grid;
            grid-template-columns: repeat(4, minmax(0, 1fr));
            gap: 8px;
            width: 100%;
        }}
        .kpi-card {{
            background: #ffffff;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 8px 10px;
            display: flex;
            flex-direction: column;
            justify-content: space-between;
            box-shadow: 0 1px 2px rgba(15, 23, 42, 0.03);
            min-width: 0;
        }}
        .kpi-label {{
            font-size: 11px;
            font-weight: 700;
            color: #475569;
            margin-bottom: 3px;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
        }}
        .kpi-val {{
            font-size: 18.5px;
            font-weight: 800;
            color: #0f172a;
            line-height: 1.15;
            letter-spacing: -0.3px;
        }}
        .kpi-sub {{
            font-size: 11px;
            color: #64748b;
            margin-top: 4px;
            font-weight: 600;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
        }}
        .maori-bars-wrap {{
            display: flex;
            flex-direction: column;
            gap: 4px;
            margin-top: 2px;
        }}
        .maori-bar-row {{
            display: flex;
            flex-direction: column;
            gap: 2px;
        }}
        .maori-bar-head {{
            display: flex;
            justify-content: space-between;
            font-size: 11px;
            font-weight: 700;
            color: #1e293b;
            white-space: nowrap;
        }}
        .maori-bar-track {{
            width: 100%;
            height: 6px;
            background: #e2e8f0;
            border-radius: 4px;
            overflow: hidden;
        }}
        .maori-bar-fill {{
            height: 100%;
            border-radius: 4px;
        }}
        /* Stack the two demographic/commuter donut charts vertically so they fit 100% within the allocated width */
        .demo-charts-grid {{
            display: grid;
            grid-template-columns: 1fr;
            gap: 10px;
            width: 100%;
        }}
        .demo-chart-card {{
            background: #ffffff;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 10px 14px;
            display: flex;
            flex-direction: column;
            min-width: 0;
            width: 100%;
            box-sizing: border-box;
        }}
        .demo-chart-card h4 {{
            margin: 0 0 6px 0;
            font-size: 13px;
            font-weight: 800;
            color: #1e293b;
            text-align: left;
        }}
        .demo-donut-row {{
            display: flex;
            align-items: center;
            justify-content: flex-start;
            gap: 18px;
            width: 100%;
            min-width: 0;
        }}
        .demo-legend-list {{
            display: flex;
            flex-direction: column;
            gap: 5px;
            flex: 1;
            min-width: 0;
        }}
        .demo-legend-item {{
            display: flex;
            align-items: center;
            justify-content: space-between;
            font-size: 12.5px;
            padding-bottom: 4px;
            border-bottom: 1px solid #f1f5f9;
            gap: 8px;
        }}
        .demo-legend-item:last-child {{
            border-bottom: none;
            padding-bottom: 0;
        }}
        .demo-legend-left {{
            display: flex;
            align-items: center;
            gap: 6px;
            color: #1e293b;
            font-weight: 600;
            white-space: nowrap;
        }}
        .demo-legend-val {{
            font-weight: 800;
            color: #0f172a;
            white-space: nowrap;
            margin-left: auto;
        }}
        .inflow-chart-card {{
            background: #ffffff;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 10px 14px;
            width: 100%;
            min-width: 0;
            box-sizing: border-box;
        }}
        .inflow-chart-header {{
            display: flex;
            align-items: center;
            justify-content: space-between;
            margin-bottom: 8px;
            flex-wrap: wrap;
            gap: 6px;
        }}
        .inflow-chart-header h4 {{
            margin: 0;
            font-size: 13px;
            font-weight: 800;
            color: #1e293b;
        }}
        .inflow-chart-sub {{
            font-size: 11px;
            color: #64748b;
            font-weight: 600;
        }}
        .inflow-bars-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(210px, 1fr));
            gap: 5px 12px;
            max-height: 185px;
            overflow-y: auto;
            padding-right: 4px;
            width: 100%;
        }}
        .inflow-bar-item {{
            display: grid;
            grid-template-columns: minmax(105px, 1.2fr) 1fr auto;
            align-items: center;
            gap: 7px;
            padding: 4px 7px;
            border-radius: 6px;
            background: #f8fafc;
            border: 1px solid #e2e8f0;
            cursor: pointer;
            transition: all 0.14s ease;
            font-size: 12px;
        }}
        .inflow-bar-item:hover {{
            background: #eff6ff;
            border-color: #6b8ead;
            transform: translateY(-1px);
        }}
        .inflow-bar-name {{
            font-weight: 700;
            color: #1e3a8a;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
        }}
        .inflow-bar-track {{
            width: 100%;
            height: 9px;
            background: #e2e8f0;
            border-radius: 4px;
            overflow: hidden;
        }}
        .inflow-bar-fill {{
            height: 100%;
            background: linear-gradient(90deg, #6b8ead 0%, #4a6fa5 100%);
            border-radius: 4px;
        }}
        .inflow-bar-count {{
            font-weight: 800;
            color: #0f172a;
            font-size: 11.5px;
            white-space: nowrap;
            text-align: right;
            min-width: 34px;
        }}
        @media (max-width: 900px) {{
            .kpi-strip {{
                grid-template-columns: repeat(2, 1fr);
            }}
            .demo-charts-grid {{
                grid-template-columns: 1fr;
            }}
        }}
        /* Side-by-Side Pie Charts Grid */
        .pies-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 16px;
        }}
        .pie-panel {{
            background: #ffffff;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 14px 14px 16px 14px;
            display: flex;
            flex-direction: column;
            align-items: center;
        }}
        .pie-panel h3 {{
            margin: 0 0 10px 0;
            font-size: 14.5px;
            font-weight: 800;
            color: #1e293b;
            text-align: center;
        }}
        .pie-svg-wrap {{
            display: flex;
            justify-content: center;
            align-items: center;
            margin-bottom: 10px;
        }}
        .ind-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 12.5px;
        }}
        .ind-table th {{
            text-align: left;
            padding: 5px 6px;
            border-bottom: 2px solid #cbd5e1;
            color: #475569;
            font-size: 11.5px;
            font-weight: 700;
        }}
        .ind-table th.num, .ind-table td.num {{
            text-align: right;
            white-space: nowrap;
        }}
        .ind-table td {{
            padding: 5px 6px;
            border-bottom: 1px solid #f1f5f9;
            color: #0f172a;
        }}
        .ind-swatch {{
            width: 12px;
            height: 12px;
            border-radius: 3px;
            display: inline-block;
            vertical-align: middle;
            margin-right: 6px;
        }}
        .hint-footer {{
            margin-top: 14px;
            font-size: 12px;
            color: #475569;
            background: #f1f5f9;
            padding: 8px 12px;
            border-radius: 6px;
        }}
        /* Right Panel: Dedicated Interactive Map */
        #map-pane {{
            display: flex;
            flex-direction: column;
            height: 100%;
            position: relative;
        }}
        .map-toolbar {{
            background: #f8fafc;
            border-bottom: 1px solid var(--border-color);
            padding: 7px 12px;
            display: flex;
            flex-wrap: wrap;
            align-items: center;
            justify-content: space-between;
            gap: 8px;
            font-size: 12px;
        }}
        .map-chk-label {{
            display: inline-flex;
            align-items: center;
            gap: 4px;
            font-weight: 600;
            color: #334155;
            cursor: pointer;
            margin-right: 6px;
        }}
        #map {{
            flex: 1;
            width: 100%;
            background: #e2e8f0;
        }}
        /* SA2 Map Labels & Colorbar */
        .owner-label-icon {{
            background: transparent;
            border: none;
        }}
        .sa2-map-label {{
            font-size: 11.5px;
            font-weight: 700;
            color: #0f172a;
            text-shadow: -1.5px -1.5px 0 #fff, 1.5px -1.5px 0 #fff, -1.5px 1.5px 0 #fff, 1.5px 1.5px 0 #fff, 0 0 4px #fff;
            white-space: nowrap;
            pointer-events: none;
            transform: translate(-50%, -50%);
            display: inline-block;
            text-align: center;
            line-height: 1.2;
        }}
        .sa2-map-label.workplace-target {{
            color: #991b1b;
            font-size: 12.5px;
            text-shadow: -1.5px -1.5px 0 #fef2f2, 1.5px -1.5px 0 #fef2f2, -1.5px 1.5px 0 #fef2f2, 1.5px 1.5px 0 #fef2f2, 0 0 6px #fff;
        }}
        .plant-label-tooltip {{
            background: rgba(255, 255, 255, 0.94) !important;
            border: 1.5px solid #dc2626 !important;
            border-radius: 4px !important;
            color: #991b1b !important;
            font-weight: 700 !important;
            font-size: 11px !important;
            padding: 2px 6px !important;
        }}
        #census-colorbar-box {{
            position: absolute;
            bottom: 28px;
            right: 14px;
            background: rgba(255, 255, 255, 0.96);
            border: 1px solid #94a3b8;
            border-radius: 8px;
            padding: 10px 14px;
            box-shadow: 0 3px 12px rgba(0,0,0,0.18);
            z-index: 1000;
            min-width: 210px;
        }}
        .cb-title {{
            font-weight: 700;
            color: #0f172a;
            margin-bottom: 6px;
            font-size: 12px;
        }}
        .cb-gradient-bar {{
            height: 14px;
            width: 100%;
            border-radius: 3px;
            border: 1px solid #64748b;
            margin-bottom: 4px;
        }}
        .cb-ticks {{
            display: flex;
            justify-content: space-between;
            font-size: 11px;
            color: #1e293b;
            font-weight: 700;
        }}
        .cb-workplace-key {{
            display: flex;
            align-items: center;
            gap: 8px;
            margin-top: 8px;
            padding-top: 7px;
            border-top: 1px solid #cbd5e1;
            font-size: 12px;
            font-weight: 700;
            color: #991b1b;
        }}
        .swatch-hatch {{
            width: 18px;
            height: 13px;
            border: 1.5px solid #dc2626;
            background: repeating-linear-gradient(
                -45deg,
                rgba(220, 38, 38, 0.14),
                rgba(220, 38, 38, 0.14) 3px,
                rgba(220, 38, 38, 0.72) 3px,
                rgba(220, 38, 38, 0.72) 5px
            );
            display: inline-block;
            border-radius: 2px;
            flex-shrink: 0;
        }}
        @media (max-width: 1150px) and (min-width: 769px) {{
            #main-split {{
                grid-template-columns: 1fr;
                grid-template-rows: 55% 45%;
            }}
            #report-pane {{
                border-right: none;
                border-bottom: 2px solid var(--border-color);
            }}
        }}

        /* ==================================================================== */
        /* RESPONSIVE DESIGN FOR IPHONE & MOBILE SCREENS (<= 768px)             */
        /* (PC / Desktop layout remains completely unchanged)                   */
        /* ==================================================================== */
        .mobile-view-tabs {{
            display: none;
        }}
        @media (max-width: 768px) {{
            #app-wrapper {{
                height: 100vh;
                height: 100dvh;
            }}
            /* Header compacting for iPhone (Header Grid with Title, Lang Toggle & App Switcher) */
            .top-header {{
                display: grid;
                grid-template-columns: 1fr auto;
                grid-template-rows: auto auto;
                grid-template-areas: 
                    "brand lang"
                    "nav   nav";
                padding: 8px 10px;
                gap: 6px 8px;
                align-items: center;
            }}
            .header-brand {{
                grid-area: brand;
                min-width: 0;
            }}
            .header-brand h1 {{
                font-size: 13.5px;
                line-height: 1.25;
                margin: 0;
                word-break: break-word;
            }}
            .header-brand p {{
                display: none;
            }}
            .header-nav-right {{
                display: contents;
            }}
            .lang-toggle {{
                grid-area: lang;
                justify-self: end;
                flex-shrink: 0;
                margin: 0;
                z-index: 1020;
            }}
            .lang-toggle-opt {{
                padding: 4px 10px;
                font-size: 11px;
                font-weight: 700;
                touch-action: manipulation;
                -webkit-tap-highlight-color: rgba(255, 255, 255, 0.2);
            }}
            .app-nav-pills {{
                grid-area: nav;
                display: flex;
                width: 100%;
                gap: 6px;
            }}
            .app-nav-btn {{
                flex: 1;
                padding: 5px 8px;
                font-size: 11px;
                font-weight: 700;
                text-align: center;
                white-space: nowrap;
                overflow: hidden;
                text-overflow: ellipsis;
            }}
            /* Toolbar & quick selection */
            .quick-toolbar {{
                padding: 6px 10px;
                gap: 8px;
                flex-direction: column;
                align-items: stretch;
            }}
            .toolbar-group {{
                display: flex;
                flex-wrap: wrap;
                align-items: center;
                gap: 5px;
                width: 100%;
            }}
            .toolbar-label {{
                font-size: 11px;
                font-weight: 700;
                color: #334155;
                display: block;
                width: 100%;
                margin-bottom: 2px;
            }}
            .select-ctrl {{
                font-size: 11.5px;
                padding: 4px 7px;
                flex: 1;
                min-width: 140px;
            }}
            .search-ctrl {{
                font-size: 11.5px;
                padding: 4px 7px;
                width: 100%;
            }}
            .chip-row {{
                display: flex;
                flex-wrap: wrap;
                gap: 5px;
                width: 100%;
                overflow: visible;
            }}
            .area-chip {{
                font-size: 11px;
                padding: 4px 8px;
                white-space: nowrap;
                flex-shrink: 0;
                border-radius: 6px;
                touch-action: manipulation;
            }}

            /* Mobile View Switcher (Report Tab vs Map Tab) */
            .mobile-view-tabs {{
                display: flex;
                background: #0f172a;
                padding: 4px 8px;
                gap: 6px;
                border-bottom: 1px solid #334155;
            }}
            .mobile-view-tab {{
                flex: 1;
                text-align: center;
                padding: 7px 10px;
                font-size: 12px;
                font-weight: 700;
                color: rgba(255, 255, 255, 0.78);
                background: rgba(255, 255, 255, 0.1);
                border: 1px solid rgba(255, 255, 255, 0.2);
                border-radius: 6px;
                cursor: pointer;
                transition: all 0.15s;
            }}
            .mobile-view-tab.active {{
                background: #0284c7;
                color: #ffffff;
                border-color: #38bdf8;
            }}

            /* Main Split in Tabbed Mobile Mode */
            #main-split {{
                grid-template-columns: 100% !important;
                grid-template-rows: 100% !important;
                width: 100% !important;
                max-width: 100vw !important;
                position: relative;
                overflow: hidden;
            }}
            #report-pane {{
                width: 100% !important;
                max-width: 100vw !important;
                height: 100%;
                padding: 10px 10px 32px 10px;
                border-right: none;
                box-sizing: border-box;
                overflow-x: hidden;
                overflow-y: auto;
                -webkit-overflow-scrolling: touch;
            }}
            #map-pane {{
                width: 100% !important;
                max-width: 100vw !important;
                height: 100%;
                display: none;
                box-sizing: border-box;
                overflow: hidden;
            }}
            #main-split.show-map #report-pane {{
                display: none;
            }}
            #main-split.show-map #map-pane {{
                display: flex;
            }}

            /* Report Sheet adjustments for iPhone */
            .report-sheet {{
                padding: 12px 10px 16px 10px;
                border-radius: 8px;
                width: 100%;
                max-width: 100%;
                box-sizing: border-box;
                overflow: hidden;
            }}
            .report-title-row {{
                gap: 8px;
            }}
            .report-title-row h2 {{
                font-size: 16px;
                word-break: break-word;
            }}
            .report-title-actions {{
                width: 100%;
                justify-content: flex-start;
                flex-wrap: wrap;
                gap: 6px;
            }}
            .report-title-actions .btn-action {{
                flex: 1;
                text-align: center;
                white-space: nowrap;
            }}
            .btn-action {{
                padding: 4px 7px;
                font-size: 11px;
            }}
            .summary-box {{
                padding: 10px 8px;
                width: 100%;
                max-width: 100%;
                box-sizing: border-box;
            }}
            .kpi-strip {{
                grid-template-columns: 1fr 1fr;
                gap: 6px;
                width: 100%;
            }}
            .kpi-card {{
                padding: 7px 8px;
            }}
            .kpi-val {{
                font-size: 16px;
            }}
            .demo-chart-card {{
                padding: 8px 10px;
                width: 100%;
                box-sizing: border-box;
            }}
            .demo-donut-row {{
                gap: 10px;
                flex-wrap: wrap;
                justify-content: center;
            }}
            .demo-legend-list {{
                width: 100%;
                min-width: 0;
            }}
            .inflow-chart-card {{
                padding: 8px 10px;
                width: 100%;
                box-sizing: border-box;
            }}
            .inflow-bars-grid {{
                grid-template-columns: 1fr;
                width: 100%;
                box-sizing: border-box;
            }}
            .inflow-bar-item {{
                grid-template-columns: minmax(85px, 1fr) 1fr auto;
                font-size: 11.5px;
                padding: 3px 6px;
            }}
            .pies-grid {{
                grid-template-columns: 1fr;
                gap: 12px;
                width: 100%;
            }}
            .pie-panel {{
                padding: 10px 6px;
                width: 100%;
                max-width: 100%;
                box-sizing: border-box;
                overflow: hidden;
            }}
            .pie-panel h3 {{
                font-size: 12.5px;
            }}
            .pie-svg-wrap svg {{
                max-width: 100%;
                height: auto;
            }}
            .ind-table {{
                font-size: 11px;
                width: 100%;
            }}
            .ind-table th, .ind-table td {{
                padding: 4px 3px;
            }}

            /* Map adjustments for iPhone */
            .map-toolbar {{
                padding: 6px 8px;
                gap: 5px;
            }}
            .map-chk-label span {{
                font-size: 11px;
            }}
            #census-colorbar-box {{
                bottom: 18px;
                right: 8px;
                padding: 7px 10px;
                min-width: 170px;
                transform: scale(0.92);
                transform-origin: bottom right;
            }}

            /* Modal Screen iPhone adaptations */
            .plant-modal-overlay {{
                padding: 8px;
                align-items: flex-end;
            }}
            .plant-modal-card {{
                max-width: 100%;
                max-height: 88vh;
                max-height: 88dvh;
                border-radius: 14px 14px 6px 6px;
            }}
            .plant-modal-header {{
                padding: 12px 14px;
            }}
            .plant-modal-title-wrap h2 {{
                font-size: 16.5px;
            }}
            .plant-modal-body {{
                padding: 12px 14px 18px 14px;
                gap: 14px;
                -webkit-overflow-scrolling: touch;
            }}
            .plant-specs-table th, .plant-specs-table td {{
                padding: 7px 9px;
                font-size: 12px;
            }}
            .plant-modal-footer {{
                padding: 8px 12px;
            }}
        }}

        @media (max-width: 720px) and (min-width: 769px) {{
            .pies-grid {{
                grid-template-columns: 1fr;
            }}
        }}
        /* Plant Case-Study Description Modal Screen */
        .plant-modal-overlay {{
            position: fixed;
            inset: 0;
            background: rgba(15, 23, 42, 0.62);
            backdrop-filter: blur(4px);
            z-index: 3500;
            display: none;
            align-items: center;
            justify-content: center;
            padding: 20px;
            opacity: 0;
            transition: opacity 0.18s ease;
        }}
        .plant-modal-overlay.open {{
            display: flex;
            opacity: 1;
        }}
        .plant-modal-card {{
            background: #ffffff;
            width: 100%;
            max-width: 860px;
            max-height: 90vh;
            border-radius: 14px;
            box-shadow: 0 24px 48px rgba(15, 23, 42, 0.35);
            border: 1px solid #cbd5e1;
            display: flex;
            flex-direction: column;
            overflow: hidden;
        }}
        .plant-modal-header {{
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
            color: #ffffff;
            padding: 18px 24px;
            display: flex;
            align-items: flex-start;
            justify-content: space-between;
            gap: 16px;
            border-bottom: 3px solid #dc2626;
        }}
        .plant-modal-title-wrap h2 {{
            font-size: 20px;
            font-weight: 800;
            color: #ffffff;
            line-height: 1.25;
            margin-bottom: 4px;
        }}
        .plant-modal-subtitle {{
            font-size: 13px;
            color: #cbd5e1;
            font-weight: 500;
        }}
        .plant-modal-close {{
            background: rgba(255, 255, 255, 0.12);
            border: 1px solid rgba(255, 255, 255, 0.25);
            color: #ffffff;
            width: 34px;
            height: 34px;
            border-radius: 8px;
            font-size: 18px;
            font-weight: 700;
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            flex-shrink: 0;
            transition: background 0.15s;
        }}
        .plant-modal-close:hover {{
            background: #dc2626;
            border-color: #dc2626;
        }}
        .plant-modal-body {{
            padding: 20px 24px 24px 24px;
            overflow-y: auto;
            display: flex;
            flex-direction: column;
            gap: 18px;
        }}
        .plant-specs-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            border: 1px solid #cbd5e1;
            border-radius: 8px;
            overflow: hidden;
        }}
        .plant-specs-table tr:nth-child(even) {{
            background: #f8fafc;
        }}
        .plant-specs-table th {{
            width: 28%;
            text-align: left;
            padding: 10px 14px;
            background: #f1f5f9;
            color: #1e293b;
            font-weight: 700;
            border-bottom: 1px solid #e2e8f0;
            border-right: 1px solid #e2e8f0;
            vertical-align: top;
        }}
        .plant-specs-table td {{
            padding: 10px 14px;
            color: #0f172a;
            line-height: 1.5;
            border-bottom: 1px solid #e2e8f0;
            vertical-align: top;
        }}
        .plant-section-block h3 {{
            font-size: 14.5px;
            font-weight: 800;
            color: #0f172a;
            margin-bottom: 8px;
            padding-bottom: 5px;
            border-bottom: 2px solid #e2e8f0;
            display: flex;
            align-items: center;
            gap: 7px;
        }}
        .plant-section-block h3::before {{
            content: '';
            display: inline-block;
            width: 4px;
            height: 15px;
            background: #dc2626;
            border-radius: 2px;
        }}
        .plant-section-block p {{
            font-size: 13.5px;
            line-height: 1.68;
            color: #1e293b;
            margin-bottom: 10px;
        }}
        .plant-section-block p:last-child {{
            margin-bottom: 0;
        }}
        .plant-takeaway-box {{
            background: #eff6ff;
            border-left: 4px solid #2563eb;
            padding: 12px 16px;
            border-radius: 0 8px 8px 0;
        }}
        .plant-takeaway-box h3 {{
            border-bottom: none;
            margin-bottom: 5px;
            padding-bottom: 0;
            color: #1e3a8a;
        }}
        .plant-takeaway-box h3::before {{
            background: #2563eb;
        }}
        .plant-takeaway-box p {{
            color: #1e3a8a;
            font-weight: 600;
        }}
        .conf-tag {{
            display: inline-block;
            font-size: 10.5px;
            font-weight: 700;
            padding: 1px 6px;
            border-radius: 4px;
            margin: 0 2px;
            vertical-align: baseline;
        }}
        .conf-confirmed {{
            background: #dcfce7;
            color: #166534;
            border: 1px solid #86efac;
        }}
        .conf-inference {{
            background: #fef3c7;
            color: #92400e;
            border: 1px solid #fde68a;
        }}
        .plant-sources-list {{
            list-style: none;
            padding: 0;
            margin: 0;
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(250px, 1fr));
            gap: 6px 12px;
        }}
        .plant-sources-list li a {{
            display: inline-flex;
            align-items: center;
            gap: 5px;
            font-size: 12px;
            color: #2563eb;
            text-decoration: none;
            padding: 4px 8px;
            border-radius: 5px;
            background: #f8fafc;
            border: 1px solid #e2e8f0;
            width: 100%;
            transition: all 0.14s;
        }}
        .plant-sources-list li a:hover {{
            background: #eff6ff;
            border-color: #93c5fd;
            text-decoration: underline;
        }}
        .plant-tags-note {{
            font-size: 11px;
            color: #64748b;
            border-top: 1px solid #e2e8f0;
            padding-top: 10px;
            font-style: italic;
        }}
        .plant-modal-footer {{
            padding: 12px 24px;
            background: #f8fafc;
            border-top: 1px solid #e2e8f0;
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 12px;
            flex-wrap: wrap;
        }}
        .plant-nav-pills {{
            display: flex;
            flex-wrap: wrap;
            gap: 5px;
        }}
        .plant-nav-pill {{
            font-size: 11.5px;
            padding: 5px 10px;
            border-radius: 6px;
            border: 1px solid #cbd5e1;
            background: #ffffff;
            color: #334155;
            cursor: pointer;
            font-weight: 600;
            transition: all 0.14s;
        }}
        .plant-nav-pill:hover {{
            border-color: #dc2626;
            color: #dc2626;
            background: #fef2f2;
        }}
        .plant-nav-pill.active {{
            background: #dc2626;
            color: #ffffff;
            border-color: #dc2626;
        }}
    </style>
</head>
<body>
<!-- SVG Pattern Definition for Workplace Diagonal Red Hatch -->
<svg width="0" height="0" style="position:absolute; pointer-events:none;">
    <defs>
        <pattern id="workplace-hatch" width="8" height="8" patternTransform="rotate(45 0 0)" patternUnits="userSpaceOnUse">
            <rect width="8" height="8" fill="rgba(254, 226, 226, 0.42)" />
            <line x1="0" y1="0" x2="0" y2="8" stroke="#dc2626" stroke-width="3.2" />
        </pattern>
    </defs>
</svg>

<div id="app-wrapper">
    <!-- Top Header Bar -->
    <header class="top-header">
        <div class="header-brand">
            <h1>{ct['app_title']}</h1>
            <p>{ct['app_sub']}</p>
        </div>
        <div class="header-nav-right">
            <div class="app-nav-pills">
                <a href="{ct['land_href']}" class="app-nav-btn">{ct['nav_land_map']}</a>
                <a href="#" class="app-nav-btn active">{ct['nav_census_map']}</a>
            </div>
            <div class="lang-toggle" role="group" aria-label="Language selector">
                <a href="census.html" class="lang-toggle-opt {active_en}" title="English">EN</a>
                <a href="census_ja.html" class="lang-toggle-opt {active_ja}" title="日本語">JA</a>
            </div>
        </div>
    </header>

    <!-- Sub-Header Area Selection & Quick Chips Bar -->
    <div class="quick-toolbar">
        <div class="toolbar-group">
            <span class="toolbar-label">{ct['select_sa2_label']}</span>
            <select id="sa2-select" class="select-ctrl" onchange="selectSA2(this.value, true)"></select>
            <input type="text" id="sa2-search" class="search-ctrl" placeholder="{ct['search_placeholder']}" oninput="filterSA2Dropdown(this.value)" />
        </div>
        <div class="toolbar-group">
            <span class="toolbar-label">{ct['quick_areas_label']}</span>
            <div class="chip-row" id="quick-chips">
                <button class="area-chip" data-sa2="188200" onclick="selectSA2('188200', true)">Wairākei-Broadlands ⚡</button>
                <button class="area-chip" data-sa2="187800" onclick="selectSA2('187800', true)">Ohakuri ⚡</button>
                <button class="area-chip" data-sa2="201800" onclick="selectSA2('201800', true)">Golden Springs (Reporoa)</button>
                <button class="area-chip" data-sa2="201700" onclick="selectSA2('201700', true)">Kaingaroa-Whakarewarewa</button>
                <button class="area-chip" data-sa2="201600" onclick="selectSA2('201600', true)">Ngakuru</button>
                <button class="area-chip" data-sa2="187701" onclick="selectSA2('187701', true)">Marotiri (Mokai) ⚡</button>
                <button class="area-chip" data-sa2="202000" onclick="selectSA2('202000', true)">Matatā-Otakiri</button>
                <button class="area-chip" data-sa2="202900" onclick="selectSA2('202900', true)">Kawerau</button>
                <button class="area-chip" data-sa2="188900" onclick="selectSA2('188900', true)">Taupō Central</button>
            </div>
        </div>
    </div>

    <!-- Mobile View Switcher (iPhone/Mobile Only) -->
    <div class="mobile-view-tabs" role="tablist">
        <button class="mobile-view-tab active" id="btn-tab-report" onclick="switchMobileView('report')">📋 { '地域レポート (Report)' if lang_code == 'ja' else 'Report View' }</button>
        <button class="mobile-view-tab" id="btn-tab-map" onclick="switchMobileView('map')">🗺️ { '通勤流動マップ (Map)' if lang_code == 'ja' else 'Map View' }</button>
    </div>

    <!-- Main Split View: Left Report Sheet (52%) + Right Map (48%) -->
    <div id="main-split">
        <!-- LEFT PANEL: Regional Socioeconomic & Industry Report -->
        <section id="report-pane">
            <div class="report-sheet">
                <div class="report-title-row">
                    <h2 id="rpt-header-title">-</h2>
                    <div class="report-title-actions">
                        <button class="btn-action" onclick="fitCommuterCatchment()">{ct['btn_fit_flow']}</button>
                        <button class="btn-action" onclick="zoomToTargetSA2()">{ct['btn_zoom_sa2']}</button>
                    </div>
                </div>

                <div id="plant-banner" class="plant-alert-banner"></div>

                <!-- 5-Line Demographic & Commuter Flow Summary Box -->
                <div class="summary-box" id="rpt-summary-box"></div>

                <!-- Two Large Pie Charts Side-by-Side + Full Industry Tables -->
                <div class="pies-grid">
                    <div class="pie-panel">
                        <h3>{ct['pie_res_title']}</h3>
                        <div class="pie-svg-wrap" id="pie-res-svg"></div>
                        <div style="width:100%;" id="pie-res-table"></div>
                    </div>
                    <div class="pie-panel">
                        <h3>{ct['pie_work_title']}</h3>
                        <div class="pie-svg-wrap" id="pie-work-svg"></div>
                        <div style="width:100%;" id="pie-work-table"></div>
                    </div>
                </div>

                <div class="hint-footer">{ct['click_map_hint']}</div>
            </div>
        </section>

        <!-- RIGHT PANEL: Dedicated Commuter Inflow & SA2 Map -->
        <section id="map-pane">
            <div class="map-toolbar">
                <div class="toolbar-group">
                    <span class="toolbar-label">{ct['map_mode_label']}</span>
                    <select id="map-mode-select" class="select-ctrl" style="font-size:12px; padding:4px 8px;" onchange="changeMapMode(this.value)">
                        <option value="commute" selected>{ct['cm_commute']}</option>
                        <option value="pop">{ct['cm_pop']}</option>
                        <option value="maori">{ct['cm_maori']}</option>
                        <option value="work_emp">{ct['cm_work_emp']}</option>
                        <option value="geothermal_ind">{ct['cm_geothermal_ind']}</option>
                    </select>
                </div>
                <div class="toolbar-group">
                    <label class="map-chk-label">
                        <input type="checkbox" id="chk-sa2-labels" checked onchange="refreshMapLabels()" />
                        <span>{ct['chk_sa2_labels']}</span>
                    </label>
                    <label class="map-chk-label">
                        <input type="checkbox" id="chk-plants" checked onchange="togglePlants(this.checked)" />
                        <span>{ct['chk_plants']}</span>
                    </label>
                    <label class="map-chk-label">
                        <input type="checkbox" id="chk-rohe" onchange="toggleRohe(this.checked)" />
                        <span>{ct['chk_rohe']}</span>
                    </label>
                    <select id="basemap-select" class="select-ctrl" style="font-size:11.5px; padding:3px 7px;" onchange="switchBaseMap(this.value)">
                        <option value="{ct['bm_esri_topo']}" selected>{ct['bm_esri_topo']}</option>
                        <option value="{ct['bm_google_road']}">{ct['bm_google_road']}</option>
                        <option value="{ct['bm_google_hybrid']}">{ct['bm_google_hybrid']}</option>
                        <option value="{ct['bm_google_terrain']}">{ct['bm_google_terrain']}</option>
                    </select>
                </div>
            </div>
            <div id="map"></div>
            <div id="census-colorbar-box"></div>
        </section>
    </div>
</div>

<!-- Plant Case-Study Description Modal Screen -->
<div id="plant-modal-overlay" class="plant-modal-overlay" onclick="if(event.target===this) closePlantModal()">
    <div class="plant-modal-card" role="dialog" aria-modal="true" aria-labelledby="plant-modal-title">
        <div class="plant-modal-header">
            <div class="plant-modal-title-wrap">
                <h2 id="plant-modal-title">-</h2>
                <div id="plant-modal-subtitle" class="plant-modal-subtitle">-</div>
            </div>
            <button class="plant-modal-close" onclick="closePlantModal()" title="Close (Esc)">✕</button>
        </div>
        <div class="plant-modal-body" id="plant-modal-body"></div>
        <div class="plant-modal-footer">
            <div class="plant-nav-pills" id="plant-modal-nav"></div>
            <button class="btn-action" onclick="closePlantModal()" style="font-weight:700;">{ '閉じる (Close)' if lang_code == 'ja' else 'Close' }</button>
        </div>
    </div>
</div>

<script>
const LANG = {json.dumps(lang_code)};
const I18N = {json.dumps(ct, ensure_ascii=False)};
const APP_CONFIG = {json.dumps(app_config)};
const PLANTS_DATA = {json.dumps(plants_list, ensure_ascii=False)};
const PRIMARY_ROHE_DATA = {json.dumps(primary_rohe_only, ensure_ascii=False)};
const CENSUS_SA2_GEOJSON = {json.dumps(census_sa2_geojson_obj, ensure_ascii=False)};

// Initialize Leaflet Map
const map = L.map('map', {{
    center: [-38.35, 176.35],
    zoom: 10,
    zoomControl: false
}});
L.control.zoom({{ position: 'topright' }}).addTo(map);
L.control.scale({{ position: 'bottomleft', metric: true, imperial: false }}).addTo(map);

const baseMaps = {{
    [I18N.bm_esri_topo]: L.tileLayer(APP_CONFIG.tileUrls.esriTopo, {{
        maxZoom: 19,
        attribution: 'Tiles &copy; Esri'
    }}),
    [I18N.bm_google_road]: L.tileLayer(APP_CONFIG.tileUrls.googleRoadmap, {{
        maxZoom: 20,
        attribution: '&copy; Google Maps'
    }}),
    [I18N.bm_google_hybrid]: L.tileLayer(APP_CONFIG.tileUrls.googleHybrid, {{
        maxZoom: 20,
        attribution: '&copy; Google Maps'
    }}),
    [I18N.bm_google_terrain]: L.tileLayer(APP_CONFIG.tileUrls.googleTerrain, {{
        maxZoom: 20,
        attribution: '&copy; Google Maps'
    }})
}};

let activeBaseLayer = baseMaps[I18N.bm_esri_topo];
activeBaseLayer.addTo(map);

function switchBaseMap(name) {{
    if (baseMaps[name]) {{
        if (activeBaseLayer) map.removeLayer(activeBaseLayer);
        activeBaseLayer = baseMaps[name];
        activeBaseLayer.addTo(map);
    }}
}}

map.createPane('rohePane');
map.getPane('rohePane').style.zIndex = 340;
map.createPane('censusPane');
map.getPane('censusPane').style.zIndex = 410;
map.createPane('censusLabelPane');
map.getPane('censusLabelPane').style.zIndex = 580;
map.createPane('plantPane');
map.getPane('plantPane').style.zIndex = 650;

const censusSvgRenderer = L.svg({{ padding: 0.5, pane: 'censusPane' }});

// Optional Primary Rohe reference layer
const roheGroup = L.layerGroup();
PRIMARY_ROHE_DATA.forEach(item => {{
    if (item.category === 'primary') {{
        L.geoJSON(item.geometry, {{
            pane: 'rohePane',
            interactive: false,
            style: {{
                color: item.style.color,
                fillColor: item.style.fillColor,
                fillOpacity: 0.08,
                weight: 2.0,
                dashArray: item.style.dashArray
            }}
        }}).addTo(roheGroup);
    }}
}});

function toggleRohe(checked) {{
    if (checked) roheGroup.addTo(map);
    else map.removeLayer(roheGroup);
}}

// Geothermal Power Stations reference layer
const plantsGroup = L.layerGroup().addTo(map);
function formatConfidenceTags(text) {{
    if (!text) return '';
    return text
        .replace(/\[(Fact\s+[^\]]+)\]/g, '<span class="conf-tag conf-confirmed">[$1]</span>')
        .replace(/\[(Inference\s+[^\]]+)\]/g, '<span class="conf-tag conf-inference">[$1]</span>');
}}

function openPlantModal(plantName) {{
    const plant = PLANTS_DATA.find(p => p.name.toLowerCase() === String(plantName).toLowerCase());
    if (!plant) return;
    const desc = (LANG === 'ja' && plant.desc_ja && plant.desc_ja.title) ? plant.desc_ja : (plant.desc_en || plant.desc_ja);
    if (!desc) return;

    document.getElementById('plant-modal-title').textContent = `⚡ ${{desc.title || plant.name}}`;
    document.getElementById('plant-modal-subtitle').textContent = desc.subtitle || `${{plant.capacity_mw}} MW — ${{plant.operator}}`;

    let specsRowsHtml = '';
    if (desc.specs && desc.specs.length > 0) {{
        desc.specs.forEach(row => {{
            specsRowsHtml += `
                <tr>
                    <th>${{row.label}}</th>
                    <td>${{formatConfidenceTags(row.value)}}</td>
                </tr>
            `;
        }});
    }}

    let sectionsHtml = '';
    if (desc.sections && desc.sections.length > 0) {{
        desc.sections.forEach(sec => {{
            const isTakeaway = sec.heading.toLowerCase().includes('takeaway') || sec.heading.includes('ポイント') || sec.heading.includes('要点');
            const itemsList = sec.items || sec.paragraphs || [];
            const parasHtml = itemsList.map(p => `<p>• ${{formatConfidenceTags(p)}}</p>`).join('');
            sectionsHtml += `
                <div class="plant-section-block ${{isTakeaway ? 'plant-takeaway-box' : ''}}">
                    <h3>${{sec.heading}}</h3>
                    ${{parasHtml}}
                </div>
            `;
        }});
    }}

    let sourcesHtml = '';
    if (desc.sources && desc.sources.length > 0) {{
        const srcHeading = LANG === 'ja' ? '情報源 (Sources)' : 'Sources';
        const itemsHtml = desc.sources.map(s => {{
            if (s.url) {{
                return `<li><a href="${{s.url}}" target="_blank" rel="noopener noreferrer">🔗 ${{s.text}}</a></li>`;
            }}
            return `<li><span style="font-size:12px; color:#475569;">• ${{s.text}}</span></li>`;
        }}).join('');
        sourcesHtml = `
            <div class="plant-section-block">
                <h3>${{srcHeading}}</h3>
                <ul class="plant-sources-list">${{itemsHtml}}</ul>
            </div>
        `;
    }}

    const tagsNoteHtml = desc.tags_note
        ? `<div class="plant-tags-note">${{formatConfidenceTags(desc.tags_note)}}</div>`
        : '';

    const bodyEl = document.getElementById('plant-modal-body');
    bodyEl.innerHTML = `
        ${{specsRowsHtml ? `<table class="plant-specs-table"><tbody>${{specsRowsHtml}}</tbody></table>` : ''}}
        ${{sectionsHtml}}
        ${{sourcesHtml}}
        ${{tagsNoteHtml}}
    `;
    bodyEl.scrollTop = 0;

    // Populate footer station switcher pills
    const navEl = document.getElementById('plant-modal-nav');
    navEl.innerHTML = PLANTS_DATA.map(pl => `
        <button class="plant-nav-pill ${{pl.name === plant.name ? 'active' : ''}}"
                onclick="openPlantModal('${{pl.name}}'); map.flyTo([${{pl.lat}}, ${{pl.lon}}], Math.max(map.getZoom(), 11), {{duration:0.6}});">
            ⚡ ${{pl.name}} (${{pl.capacity_mw}} MW)
        </button>
    `).join('');

    document.getElementById('plant-modal-overlay').classList.add('open');
}}

function closePlantModal() {{
    const overlay = document.getElementById('plant-modal-overlay');
    if (overlay) overlay.classList.remove('open');
}}

document.addEventListener('keydown', (e) => {{
    if (e.key === 'Escape') closePlantModal();
}});

PLANTS_DATA.forEach(plant => {{
    const icon = L.divIcon({{
        className: '',
        html: `<div style="width:0;height:0;border-left:9px solid transparent;border-right:9px solid transparent;border-bottom:16px solid #dc2626;filter:drop-shadow(0 1px 2px rgba(0,0,0,0.45));cursor:pointer;" title="${{plant.name}}"></div>`,
        iconSize: [18, 16],
        iconAnchor: [9, 12]
    }});
    const m = L.marker([plant.lat, plant.lon], {{ icon, pane: 'plantPane' }});
    const clickHint = LANG === 'ja' ? 'クリックして詳細ノートを表示' : 'Click for plant description';
    m.bindTooltip(`⚡ ${{plant.name}} (${{plant.capacity_mw}} MW) — <span style="font-weight:500;color:#fca5a5;">${{clickHint}}</span>`, {{
        permanent: false,
        direction: 'top',
        className: 'plant-label-tooltip'
    }});
    m.on('click', () => {{
        openPlantModal(plant.name);
    }});
    m.addTo(plantsGroup);
}});

function togglePlants(checked) {{
    if (checked) plantsGroup.addTo(map);
    else map.removeLayer(plantsGroup);
}}

// Build SA2 lookup and dropdown options
let selectedSA2Code = '202000'; // Default to Matatā-Otakiri (from the user's report screenshot)
let currentMapMode = 'commute';
const sa2FeatureByCode = {{}};
const sa2LayerByCode = {{}};
const sortedCoreFeatures = [];
const sortedCatchmentFeatures = [];

CENSUS_SA2_GEOJSON.features.forEach(feat => {{
    const p = feat.properties;
    sa2FeatureByCode[p.sa2_code] = feat;
    if (p.is_core) sortedCoreFeatures.push(feat);
    else sortedCatchmentFeatures.push(feat);
}});

sortedCoreFeatures.sort((a, b) => a.properties.sa2_name.localeCompare(b.properties.sa2_name));
sortedCatchmentFeatures.sort((a, b) => a.properties.sa2_name.localeCompare(b.properties.sa2_name));

function populateSA2Dropdown(filterQuery = '') {{
    const sel = document.getElementById('sa2-select');
    const q = filterQuery.trim().toLowerCase();
    sel.innerHTML = '';

    const coreGrp = document.createElement('optgroup');
    coreGrp.label = LANG === 'ja' ? '調査対象・周辺地域 (Core Study SA2)' : 'Core Study Region SA2s';
    let firstMatchCode = null;

    sortedCoreFeatures.forEach(feat => {{
        const p = feat.properties;
        if (q && !p.sa2_name.toLowerCase().includes(q) && !p.sa2_code.includes(q)) return;
        if (!firstMatchCode) firstMatchCode = p.sa2_code;
        const opt = document.createElement('option');
        opt.value = p.sa2_code;
        const pTag = (p.plants && p.plants.length > 0) ? ' ⚡' : '';
        opt.textContent = `${{p.sa2_name}} (${{p.sa2_code}})${{pTag}} — Pop: ${{p.pop_usual.toLocaleString()}}`;
        if (p.sa2_code === selectedSA2Code) opt.selected = true;
        coreGrp.appendChild(opt);
    }});
    if (coreGrp.children.length > 0) sel.appendChild(coreGrp);

    const extGrp = document.createElement('optgroup');
    extGrp.label = LANG === 'ja' ? '広域通勤流入元地域 (Wider Catchment SA2)' : 'Wider Commuter Catchment SA2s';
    sortedCatchmentFeatures.forEach(feat => {{
        const p = feat.properties;
        if (q && !p.sa2_name.toLowerCase().includes(q) && !p.sa2_code.includes(q)) return;
        if (!firstMatchCode) firstMatchCode = p.sa2_code;
        const opt = document.createElement('option');
        opt.value = p.sa2_code;
        opt.textContent = `${{p.sa2_name}} (${{p.sa2_code}}) — Pop: ${{p.pop_usual.toLocaleString()}}`;
        if (p.sa2_code === selectedSA2Code) opt.selected = true;
        extGrp.appendChild(opt);
    }});
    if (extGrp.children.length > 0) sel.appendChild(extGrp);

    if (q && firstMatchCode && firstMatchCode !== selectedSA2Code) {{
        selectSA2(firstMatchCode, true, false);
        sel.value = firstMatchCode;
    }}
}}

function filterSA2Dropdown(val) {{
    populateSA2Dropdown(val);
}}

const BLUES_PALETTE = ['#f7fbff', '#deebf7', '#c6dbef', '#9ecae1', '#6baed6', '#4292c6', '#2171b5', '#08519c', '#08306b'];
const PURPLES_PALETTE = ['#fcfbfd', '#efedf5', '#dadaeb', '#bcbddc', '#9e9ac8', '#807dba', '#6a51a3', '#54278f', '#3f007d'];
const GREENS_PALETTE = ['#f7fcf5', '#e5f5e0', '#c7e9c0', '#a1d99b', '#74c476', '#41ab5d', '#238b45', '#006d2c', '#00441b'];
const ORANGES_PALETTE = ['#fff5eb', '#fee6ce', '#fdd0a2', '#fdae6b', '#fd8d3c', '#f16913', '#d94801', '#a63603', '#7f2704'];

function samplePalette(palette, t) {{
    const clamped = Math.max(0, Math.min(1, t));
    const idx = Math.min(palette.length - 1, Math.floor(clamped * (palette.length - 1)));
    return palette[idx];
}}

function getActiveInflowMap() {{
    const feat = sa2FeatureByCode[selectedSA2Code];
    const originMap = {{}};
    let maxInflow = 15;
    if (feat && feat.properties.inflow_origins) {{
        feat.properties.inflow_origins.forEach(o => {{
            originMap[o.code] = o.count;
            if (o.count > maxInflow) maxInflow = o.count;
        }});
    }}
    return {{ originMap, maxInflow }};
}}

function getSA2PolygonStyle(feature) {{
    const p = feature.properties;
    const isSelected = (p.sa2_code === selectedSA2Code);

    if (currentMapMode === 'commute') {{
        const {{ originMap, maxInflow }} = getActiveInflowMap();
        if (isSelected) {{
            return {{
                color: '#dc2626',
                weight: 3.2,
                fillColor: 'url(#workplace-hatch)',
                fillOpacity: 0.95
            }};
        }}
        const inflowCount = originMap[p.sa2_code] || 0;
        if (inflowCount > 0) {{
            const ratio = Math.min(1.0, inflowCount / maxInflow);
            return {{
                color: '#1e3a8a',
                weight: 2.0,
                fillColor: samplePalette(BLUES_PALETTE, Math.max(0.20, ratio)),
                fillOpacity: 0.80
            }};
        }}
        return {{
            color: p.is_core ? '#94a3b8' : '#cbd5e1',
            weight: p.is_core ? 1.0 : 0.6,
            fillColor: '#ffffff',
            fillOpacity: p.is_core ? 0.25 : 0.08
        }};
    }}

    if (!p.is_core && !isSelected) {{
        return {{
            color: '#cbd5e1',
            weight: 0.5,
            fillColor: '#f8fafc',
            fillOpacity: 0.1
        }};
    }}

    let ratio = 0;
    let palette = BLUES_PALETTE;
    if (currentMapMode === 'pop') {{
        ratio = Math.min(1.0, p.pop_usual / 4500.0);
        palette = BLUES_PALETTE;
    }} else if (currentMapMode === 'maori') {{
        ratio = Math.min(1.0, p.maori_eth_pct / 85.0);
        palette = PURPLES_PALETTE;
    }} else if (currentMapMode === 'work_emp') {{
        ratio = Math.min(1.0, p.work_emp_total / 2500.0);
        palette = GREENS_PALETTE;
    }} else if (currentMapMode === 'geothermal_ind') {{
        ratio = Math.min(1.0, p.work_egww_mining_pct / 25.0);
        palette = ORANGES_PALETTE;
    }}

    return {{
        color: isSelected ? '#dc2626' : '#475569',
        weight: isSelected ? 3.2 : 1.1,
        fillColor: samplePalette(palette, ratio),
        fillOpacity: 0.78
    }};
}}

const labelsGroup = L.layerGroup().addTo(map);

const sa2GeoJsonLayer = L.geoJSON(CENSUS_SA2_GEOJSON, {{
    pane: 'censusPane',
    renderer: censusSvgRenderer,
    style: getSA2PolygonStyle,
    onEachFeature: (feature, layer) => {{
        const p = feature.properties;
        sa2LayerByCode[p.sa2_code] = layer;
        layer.on('click', () => {{
            selectSA2(p.sa2_code, false);
        }});
        layer.on('mouseover', function() {{
            if (p.sa2_code !== selectedSA2Code) {{
                this.setStyle({{ weight: 2.5, color: '#0f172a' }});
            }}
        }});
        layer.on('mouseout', function() {{
            sa2GeoJsonLayer.resetStyle(this);
        }});

        const ttHtml = LANG === 'ja'
            ? `<strong>${{p.sa2_name}} (${{p.sa2_code}})</strong><br/>通常居住人口: <b>${{p.pop_usual.toLocaleString()}}人</b> | 夜間: ${{p.pop_night.toLocaleString()}}人<br/>Māori民族: <b>${{p.maori_eth_pct.toFixed(1)}}%</b> | 血統: ${{p.maori_desc_pct.toFixed(1)}}%<br/>就業者総数: <b>${{p.work_emp_total.toLocaleString()}}人</b> (域外流入: ${{p.commute_inflow.toLocaleString()}}人 / ${{p.commute_inflow_pct.toFixed(1)}}%)`
            : `<strong>${{p.sa2_name}} (${{p.sa2_code}})</strong><br/>Usual Resident Pop: <b>${{p.pop_usual.toLocaleString()}}</b> | Night: ${{p.pop_night.toLocaleString()}}<br/>Māori Ethnicity: <b>${{p.maori_eth_pct.toFixed(1)}}%</b> | Descent: ${{p.maori_desc_pct.toFixed(1)}}%<br/>Workplace Employed: <b>${{p.work_emp_total.toLocaleString()}}</b> (Inflow: ${{p.commute_inflow.toLocaleString()}} / ${{p.commute_inflow_pct.toFixed(1)}}%)`;
        layer.bindTooltip(ttHtml, {{ sticky: true }});
    }}
}}).addTo(map);

// Large, High-Resolution SVG Pie Chart (Muted, Easy-to-Read Palette) + Full Industry Table
// Muted palette: Soft Slate Blue, Muted Sage Green, Warm Sand/Ochre, Dusty Rose/Terracotta, Muted Lavender, Soft Warm Grey
const PIE_COLORS = ['#6b8ead', '#7ba68a', '#d9a473', '#c98484', '#9a8bb5', '#d5dde6'];

function renderPieAndTable(items, svgContainerId, tableContainerId) {{
    const svgBox = document.getElementById(svgContainerId);
    const tblBox = document.getElementById(tableContainerId);
    if (!items || items.length === 0) {{
        svgBox.innerHTML = `<div style="padding:45px 12px; color:#64748b; font-size:13px; text-align:center;">${{I18N.no_ind_data}}</div>`;
        tblBox.innerHTML = '';
        return;
    }}

    const size = 228;
    const cx = size / 2;
    const cy = size / 2;
    const r = 96;
    let currentAngle = -Math.PI / 2;

    let pathsHtml = '';
    let labelsHtml = '';
    let rowsHtml = '';

    const totalCount = items.reduce((acc, it) => acc + it.count, 0) || 1;
    const unitSuffix = LANG === 'ja' ? '人' : '';

    items.forEach((item, idx) => {{
        const color = item.is_other ? '#d5dde6' : PIE_COLORS[idx % 5];
        const name = LANG === 'ja' ? item.ja : item.en;
        const sliceFrac = Math.max(0, item.count / totalCount);
        const sliceAngle = sliceFrac * 2 * Math.PI;
        const endAngle = currentAngle + sliceAngle;

        if (sliceFrac >= 0.999) {{
            pathsHtml += `<circle cx="${{cx}}" cy="${{cy}}" r="${{r}}" fill="${{color}}" stroke="#ffffff" stroke-width="2.2"><title>${{name}}: ${{item.count}}${{unitSuffix}} (${{item.pct.toFixed(1)}}%)</title></circle>`;
        }} else if (sliceAngle > 0.001) {{
            const x1 = cx + r * Math.cos(currentAngle);
            const y1 = cy + r * Math.sin(currentAngle);
            const x2 = cx + r * Math.cos(endAngle);
            const y2 = cy + r * Math.sin(endAngle);
            const largeArc = sliceAngle > Math.PI ? 1 : 0;
            const d = `M ${{cx}} ${{cy}} L ${{x1.toFixed(2)}} ${{y1.toFixed(2)}} A ${{r}} ${{r}} 0 ${{largeArc}} 1 ${{x2.toFixed(2)}} ${{y2.toFixed(2)}} Z`;
            pathsHtml += `<path d="${{d}}" fill="${{color}}" stroke="#ffffff" stroke-width="2.2"><title>${{name}}: ${{item.count}}${{unitSuffix}} (${{item.pct.toFixed(1)}}%)</title></path>`;
        }}

        if (item.pct >= 3.2) {{
            const midAngle = currentAngle + sliceAngle / 2;
            // Stagger label radius slightly on small slices so adjacent labels never crowd each other
            const radiusFactor = item.pct < 7.5 ? (idx % 2 === 0 ? 0.74 : 0.58) : 0.63;
            const lr = r * radiusFactor;
            const lx = cx + lr * Math.cos(midAngle);
            const ly = cy + lr * Math.sin(midAngle);
            labelsHtml += `<text x="${{lx.toFixed(1)}}" y="${{ly.toFixed(1)}}" fill="#0f172a" font-size="12" font-weight="800" text-anchor="middle" dominant-baseline="central" style="pointer-events:none; paint-order:stroke; stroke:rgba(255,255,255,0.92); stroke-width:3.5px; stroke-linejoin:round;">${{item.pct.toFixed(1)}}%</text>`;
        }}

        rowsHtml += `
            <tr>
                <td><span class="ind-swatch" style="background:${{color}}; border:1px solid rgba(15,23,42,0.18);"></span><strong>${{name}}</strong></td>
                <td class="num">${{item.count.toLocaleString()}}${{unitSuffix}}</td>
                <td class="num"><strong>${{item.pct.toFixed(1)}}%</strong></td>
            </tr>
        `;

        currentAngle = endAngle;
    }});

    svgBox.innerHTML = `<svg width="${{size}}" height="${{size}}" viewBox="0 0 ${{size}} ${{size}}">${{pathsHtml}}${{labelsHtml}}</svg>`;
    tblBox.innerHTML = `
        <table class="ind-table">
            <thead>
                <tr>
                    <th>${{I18N.tbl_col_industry}}</th>
                    <th class="num">${{I18N.tbl_col_count}}</th>
                    <th class="num">${{I18N.tbl_col_pct}}</th>
                </tr>
            </thead>
            <tbody>${{rowsHtml}}</tbody>
        </table>
    `;
}}

// Helper to render a clean, muted SVG Donut / Pie Chart for Demographic & Commuter Summary Cards
function buildMiniDonutSvg(slices, centerTopText, centerSubText) {{
    const size = 136;
    const cx = size / 2;
    const cy = size / 2;
    const r = 58;
    const innerR = 27;
    const total = slices.reduce((acc, s) => acc + Math.max(0, s.val), 0);
    if (total <= 0) {{
        return `<svg width="${{size}}" height="${{size}}" viewBox="0 0 ${{size}} ${{size}}">
            <circle cx="${{cx}}" cy="${{cy}}" r="${{r}}" fill="#e2e8f0" />
            <circle cx="${{cx}}" cy="${{cy}}" r="${{innerR}}" fill="#ffffff" />
            <text x="${{cx}}" y="${{cy}}" text-anchor="middle" dominant-baseline="central" font-size="11" font-weight="700" fill="#64748b">N/A</text>
        </svg>`;
    }}

    let currentAngle = -Math.PI / 2;
    let pathsHtml = '';
    let labelsHtml = '';

    slices.forEach((s) => {{
        const val = Math.max(0, s.val);
        const frac = val / total;
        const angle = frac * 2 * Math.PI;
        const endAngle = currentAngle + angle;

        if (frac >= 0.999) {{
            pathsHtml += `<circle cx="${{cx}}" cy="${{cy}}" r="${{r}}" fill="${{s.color}}" stroke="#ffffff" stroke-width="2"><title>${{s.label}}: ${{s.display}}</title></circle>`;
        }} else if (angle > 0.001) {{
            const x1 = cx + r * Math.cos(currentAngle);
            const y1 = cy + r * Math.sin(currentAngle);
            const x2 = cx + r * Math.cos(endAngle);
            const y2 = cy + r * Math.sin(endAngle);
            const largeArc = angle > Math.PI ? 1 : 0;
            const d = `M ${{cx}} ${{cy}} L ${{x1.toFixed(2)}} ${{y1.toFixed(2)}} A ${{r}} ${{r}} 0 ${{largeArc}} 1 ${{x2.toFixed(2)}} ${{y2.toFixed(2)}} Z`;
            pathsHtml += `<path d="${{d}}" fill="${{s.color}}" stroke="#ffffff" stroke-width="2"><title>${{s.label}}: ${{s.display}}</title></path>`;
        }}

        if (frac >= 0.07) {{
            const mid = currentAngle + angle / 2;
            const lr = (r + innerR) / 2;
            const lx = cx + lr * Math.cos(mid);
            const ly = cy + lr * Math.sin(mid);
            const slicePctStr = s.pctText || `${{(frac * 100).toFixed(1)}}%`;
            labelsHtml += `<text x="${{lx.toFixed(1)}}" y="${{ly.toFixed(1)}}" fill="#0f172a" font-size="10.5" font-weight="800" text-anchor="middle" dominant-baseline="central" style="pointer-events:none; paint-order:stroke; stroke:rgba(255,255,255,0.95); stroke-width:3.2px; stroke-linejoin:round;">${{slicePctStr}}</text>`;
        }}

        currentAngle = endAngle;
    }});

    const centerHtml = `
        <circle cx="${{cx}}" cy="${{cy}}" r="${{innerR}}" fill="#ffffff" stroke="#e2e8f0" stroke-width="1" />
        <text x="${{cx}}" y="${{cy - 4}}" text-anchor="middle" dominant-baseline="central" font-size="10.5" font-weight="800" fill="#0f172a">${{centerTopText}}</text>
        <text x="${{cx}}" y="${{cy + 7.5}}" text-anchor="middle" dominant-baseline="central" font-size="8" font-weight="700" fill="#64748b">${{centerSubText}}</text>
    `;

    return `<svg width="${{size}}" height="${{size}}" viewBox="0 0 ${{size}} ${{size}}" style="flex-shrink:0;">${{pathsHtml}}${{centerHtml}}${{labelsHtml}}</svg>`;
}}

function updateReportSheet() {{
    const feat = sa2FeatureByCode[selectedSA2Code];
    if (!feat) return;
    const p = feat.properties;

    document.getElementById('rpt-header-title').textContent = `${{p.sa2_name}} (${{p.sa2_code}})`;

    const plantBanner = document.getElementById('plant-banner');
    if (p.plants && p.plants.length > 0) {{
        plantBanner.style.display = 'block';
        const btnNoteLabel = LANG === 'ja' ? '詳細ノートを見る' : 'View Description';
        const plantBtns = p.plants.map(pStr => {{
            const baseName = pStr.split(' (')[0].trim();
            return `<button onclick="openPlantModal('${{baseName}}')" style="margin-left:6px; padding:2px 9px; font-size:11.5px; font-weight:700; color:#ffffff; background:#dc2626; border:none; border-radius:5px; cursor:pointer;">⚡ ${{pStr}} — ${{btnNoteLabel}}</button>`;
        }}).join(' ');
        plantBanner.innerHTML = `<span>${{I18N.plant_badge_prefix}}</span> ${{plantBtns}}`;
    }} else {{
        plantBanner.style.display = 'none';
    }}

    const isJa = (LANG === 'ja');
    const unitP = isJa ? '人' : '';

    // 1. Age Structure Donut Chart (Muted Palette: Sage Green, Slate Blue, Warm Ochre)
    const ageSlices = [
        {{
            label: isJa ? '年少人口 (0–14歳)' : 'Youth (0–14 yrs)',
            val: p.age_0_14_pct,
            pctText: `${{p.age_0_14_pct.toFixed(1)}}%`,
            display: `${{p.age_0_14_pct.toFixed(1)}}%`,
            color: '#7ba68a'
        }},
        {{
            label: isJa ? '生産年齢人口 (15–64歳)' : 'Working Age (15–64 yrs)',
            val: p.age_15_64_pct,
            pctText: `${{p.age_15_64_pct.toFixed(1)}}%`,
            display: `${{p.age_15_64_pct.toFixed(1)}}%`,
            color: '#6b8ead'
        }},
        {{
            label: isJa ? '老年人口 (65歳以上)' : 'Seniors (65+ yrs)',
            val: p.age_65_plus_pct,
            pctText: `${{p.age_65_plus_pct.toFixed(1)}}%`,
            display: `${{p.age_65_plus_pct.toFixed(1)}}%`,
            color: '#d9a473'
        }}
    ];
    const ageDonutSvg = buildMiniDonutSvg(
        ageSlices,
        `${{p.median_age.toFixed(1)}}${{isJa ? '歳' : 'y'}}`,
        isJa ? '年齢中央値' : 'Median Age'
    );
    const ageLegendHtml = ageSlices.map(s => `
        <div class="demo-legend-item">
            <span class="demo-legend-left"><span class="ind-swatch" style="background:${{s.color}}; border:1px solid rgba(15,23,42,0.18);"></span>${{s.label}}</span>
            <span class="demo-legend-val">${{s.display}}</span>
        </div>
    `).join('');

    // 2. Commuter Flow Donut Chart (Internal vs External Inflow)
    const internalPct = p.commute_base > 0 ? (p.commute_internal / p.commute_base * 100.0) : 0.0;
    const inflowPct = p.commute_base > 0 ? p.commute_inflow_pct : 0.0;
    const commuteSlices = [
        {{
            label: isJa ? '域外流入 (通勤流入)' : 'External Inflow',
            val: p.commute_inflow,
            pctText: `${{inflowPct.toFixed(1)}}%`,
            display: `${{p.commute_inflow.toLocaleString()}}${{unitP}} (${{inflowPct.toFixed(1)}}%)`,
            color: '#c98484'
        }},
        {{
            label: isJa ? '居住地内 (域内就業)' : 'Local / Internal',
            val: p.commute_internal,
            pctText: `${{internalPct.toFixed(1)}}%`,
            display: `${{p.commute_internal.toLocaleString()}}${{unitP}} (${{internalPct.toFixed(1)}}%)`,
            color: '#6b8ead'
        }}
    ];
    const commuteDonutSvg = buildMiniDonutSvg(
        commuteSlices,
        `${{p.commute_base.toLocaleString()}}${{unitP}}`,
        isJa ? '通勤流動計' : 'Flow Base'
    );
    const commuteLegendHtml = commuteSlices.map(s => `
        <div class="demo-legend-item">
            <span class="demo-legend-left"><span class="ind-swatch" style="background:${{s.color}}; border:1px solid rgba(15,23,42,0.18);"></span>${{s.label}}</span>
            <span class="demo-legend-val">${{s.display}}</span>
        </div>
    `).join('') + `
        <div class="demo-legend-item" style="margin-top:2px; padding-top:4px; border-top:1px dashed #cbd5e1;">
            <span class="demo-legend-left" style="color:#475569;">${{isJa ? '勤務地就業者総数 (全体)' : 'Total Workplace Employed'}}</span>
            <span class="demo-legend-val">${{p.work_emp_total.toLocaleString()}}${{unitP}}</span>
        </div>
    `;

    // 3. Ranked Horizontal Bar Chart for Commuter Inflow Origins
    let originsBarChartHtml = '';
    if (p.inflow_origins && p.inflow_origins.length > 0) {{
        const maxOriginCount = Math.max(...p.inflow_origins.map(o => o.count), 1);
        const totalInflow = Math.max(p.commute_inflow, 1);
        const barsHtml = p.inflow_origins.map((o, idx) => {{
            const widthPct = Math.max(6, Math.round((o.count / maxOriginCount) * 100));
            const shareOfInflow = ((o.count / totalInflow) * 100).toFixed(1);
            return `
                <div class="inflow-bar-item" onclick="selectSA2('${{o.code}}', true)" title="${{o.name}} (SA2 ${{o.code}}): ${{o.count}}${{unitP}} (${{shareOfInflow}}% of inflow)">
                    <span class="inflow-bar-name">${{idx + 1}}. ${{o.name}}</span>
                    <div class="inflow-bar-track">
                        <div class="inflow-bar-fill" style="width:${{widthPct}}%;"></div>
                    </div>
                    <span class="inflow-bar-count">${{o.count}}${{unitP}}</span>
                </div>
            `;
        }}).join('');
        originsBarChartHtml = `<div class="inflow-bars-grid">${{barsHtml}}</div>`;
    }} else {{
        originsBarChartHtml = `<div style="color:#64748b; font-size:12.5px; padding:4px 0;">${{I18N.none_inflow}}</div>`;
    }}

    const summaryBox = document.getElementById('rpt-summary-box');
    summaryBox.innerHTML = `
        <!-- Top Row: 4 KPI Cards -->
        <div class="kpi-strip">
            <div class="kpi-card">
                <div class="kpi-label">${{isJa ? '通常居住人口' : 'Usual Resident Pop'}}</div>
                <div class="kpi-val">${{p.pop_usual.toLocaleString()}}<span style="font-size:13px; font-weight:700; margin-left:2px;">${{unitP}}</span></div>
                <div class="kpi-sub">${{isJa ? '夜間人口総数' : 'Night Pop'}}: <strong>${{p.pop_night.toLocaleString()}}${{unitP}}</strong></div>
            </div>
            <div class="kpi-card">
                <div class="kpi-label">${{isJa ? '年齢中央値' : 'Median Age'}}</div>
                <div class="kpi-val">${{p.median_age.toFixed(1)}}<span style="font-size:13px; font-weight:700; margin-left:2px;">${{isJa ? '歳' : ' yrs'}}</span></div>
                <div class="kpi-sub">${{isJa ? '生産年齢' : 'Working Age'}}: <strong>${{p.age_15_64_pct.toFixed(1)}}%</strong></div>
            </div>
            <div class="kpi-card">
                <div class="kpi-label">${{isJa ? 'Māori 民族・血統' : 'Māori Share'}}</div>
                <div class="maori-bars-wrap">
                    <div class="maori-bar-row">
                        <div class="maori-bar-head"><span>${{isJa ? '民族割合' : 'Ethnicity'}}</span><span>${{p.maori_eth_pct.toFixed(1)}}%</span></div>
                        <div class="maori-bar-track"><div class="maori-bar-fill" style="width:${{Math.min(100, p.maori_eth_pct)}}%; background:#9a8bb5;"></div></div>
                    </div>
                    <div class="maori-bar-row">
                        <div class="maori-bar-head"><span>${{isJa ? '血統保有率' : 'Descent'}}</span><span>${{p.maori_desc_pct.toFixed(1)}}%</span></div>
                        <div class="maori-bar-track"><div class="maori-bar-fill" style="width:${{Math.min(100, p.maori_desc_pct)}}%; background:#6b8ead;"></div></div>
                    </div>
                </div>
            </div>
            <div class="kpi-card">
                <div class="kpi-label">${{isJa ? '勤務地就業者総数' : 'Workplace Employed'}}</div>
                <div class="kpi-val">${{p.work_emp_total.toLocaleString()}}<span style="font-size:13px; font-weight:700; margin-left:2px;">${{unitP}}</span></div>
                <div class="kpi-sub">${{isJa ? '域外流入率' : 'Inflow Rate'}}: <strong style="color:#b91c1c;">${{p.commute_inflow_pct.toFixed(1)}}%</strong> (${{p.commute_inflow.toLocaleString()}}${{unitP}})</div>
            </div>
        </div>

        <!-- Middle Row: 2 Side-by-Side Donut Charts (Age Structure & Commuter Flow) -->
        <div class="demo-charts-grid">
            <div class="demo-chart-card">
                <h4>${{isJa ? '■ 年齢構造 (Age Structure)' : '■ Age Structure'}}</h4>
                <div class="demo-donut-row">
                    ${{ageDonutSvg}}
                    <div class="demo-legend-list">${{ageLegendHtml}}</div>
                </div>
            </div>
            <div class="demo-chart-card">
                <h4>${{isJa ? '■ 通勤流動内訳 (居住地内 vs 域外流入)' : '■ Commuter Flow (Local vs Inflow)'}}</h4>
                <div class="demo-donut-row">
                    ${{commuteDonutSvg}}
                    <div class="demo-legend-list">${{commuteLegendHtml}}</div>
                </div>
            </div>
        </div>

        <!-- Bottom Row: Ranked Horizontal Bar Chart of Inflow Origins -->
        <div class="inflow-chart-card">
            <div class="inflow-chart-header">
                <h4>${{isJa ? '■ 流入元地域（域外からの通勤流入数ランキング）' : '■ Commuter Inflow Origins (Ranked by Commuter Count)'}}</h4>
                <span class="inflow-chart-sub">${{isJa ? `クリックで該当地域へ移動（計 ${{p.inflow_origins ? p.inflow_origins.length : 0}} 地域 / 流入計 ${{p.commute_inflow.toLocaleString()}}人）` : `Click any bar to inspect origin SA2 (${{p.inflow_origins ? p.inflow_origins.length : 0}} areas / ${{p.commute_inflow.toLocaleString()}} commuters)`}}</span>
            </div>
            ${{originsBarChartHtml}}
        </div>
    `;

    renderPieAndTable(p.res_top5, 'pie-res-svg', 'pie-res-table');
    renderPieAndTable(p.work_top5, 'pie-work-svg', 'pie-work-table');
}}

function refreshMapLabels() {{
    labelsGroup.clearLayers();
    if (!document.getElementById('chk-sa2-labels').checked) return;

    const targetFeat = sa2FeatureByCode[selectedSA2Code];
    const {{ originMap }} = getActiveInflowMap();

    if (currentMapMode === 'commute') {{
        if (targetFeat) {{
            const tp = targetFeat.properties;
            const icon = L.divIcon({{
                className: 'owner-label-icon',
                html: `<span class="sa2-map-label workplace-target">${{I18N.workplace_prefix}}: ${{tp.sa2_name}}</span>`,
                iconSize: [0, 0]
            }});
            L.marker([tp.rep_lat, tp.rep_lon], {{ icon, interactive: false, pane: 'censusLabelPane' }}).addTo(labelsGroup);
        }}
        Object.keys(originMap).forEach(ocode => {{
            const ofeat = sa2FeatureByCode[ocode];
            if (!ofeat) return;
            const op = ofeat.properties;
            const cnt = originMap[ocode];
            const icon = L.divIcon({{
                className: 'owner-label-icon',
                html: `<span class="sa2-map-label">${{op.sa2_name}}<br/>(${{cnt}})</span>`,
                iconSize: [0, 0]
            }});
            L.marker([op.rep_lat, op.rep_lon], {{ icon, interactive: false, pane: 'censusLabelPane' }}).addTo(labelsGroup);
        }});
    }} else {{
        if (map.getZoom() < 10) return;
        const bounds = map.getBounds();
        sortedCoreFeatures.forEach(feat => {{
            const p = feat.properties;
            if (!bounds.contains([p.rep_lat, p.rep_lon])) return;
            let valStr = '';
            if (currentMapMode === 'pop') valStr = p.pop_usual.toLocaleString();
            else if (currentMapMode === 'maori') valStr = p.maori_eth_pct.toFixed(1) + '%';
            else if (currentMapMode === 'work_emp') valStr = p.work_emp_total.toLocaleString();
            else if (currentMapMode === 'geothermal_ind') valStr = p.work_egww_mining_pct.toFixed(1) + '%';
            const isSel = (p.sa2_code === selectedSA2Code);
            const icon = L.divIcon({{
                className: 'owner-label-icon',
                html: `<span class="sa2-map-label ${{isSel ? 'workplace-target' : ''}}">${{p.sa2_name}}<br/>(${{valStr}})</span>`,
                iconSize: [0, 0]
            }});
            L.marker([p.rep_lat, p.rep_lon], {{ icon, interactive: false, pane: 'censusLabelPane' }}).addTo(labelsGroup);
        }});
    }}
}}

map.on('moveend', () => {{
    if (currentMapMode !== 'commute') refreshMapLabels();
}});

function updateColorbar() {{
    const box = document.getElementById('census-colorbar-box');
    const feat = sa2FeatureByCode[selectedSA2Code];
    const targetName = feat ? feat.properties.sa2_name : '';

    if (currentMapMode === 'commute') {{
        const {{ maxInflow }} = getActiveInflowMap();
        const midVal = Math.round(maxInflow / 2);
        box.innerHTML = `
            <div class="cb-title">${{I18N.cb_commuters}}</div>
            <div class="cb-gradient-bar" style="background:linear-gradient(to right, #f7fbff, #c6dbef, #6baed6, #2171b5, #08306b);"></div>
            <div class="cb-ticks"><span>1</span><span>${{midVal}}</span><span>${{maxInflow}}</span></div>
            <div class="cb-workplace-key">
                <span class="swatch-hatch"></span>
                <span>${{I18N.workplace_prefix}}: ${{targetName}}</span>
            </div>
        `;
        return;
    }}

    let title = I18N.cb_pop;
    let grad = 'linear-gradient(to right, #f7fbff, #6baed6, #08306b)';
    let minS = '0', midS = '2,250', maxS = '4,500+';
    if (currentMapMode === 'maori') {{
        title = I18N.cb_maori;
        grad = 'linear-gradient(to right, #fcfbfd, #9e9ac8, #3f007d)';
        minS = '0%'; midS = '42%'; maxS = '85%+';
    }} else if (currentMapMode === 'work_emp') {{
        title = I18N.cb_work_emp;
        grad = 'linear-gradient(to right, #f7fcf5, #74c476, #00441b)';
        minS = '0'; midS = '1,250'; maxS = '2,500+';
    }} else if (currentMapMode === 'geothermal_ind') {{
        title = I18N.cb_geo_ind;
        grad = 'linear-gradient(to right, #fff5eb, #fd8d3c, #7f2704)';
        minS = '0%'; midS = '12%'; maxS = '25%+';
    }}
    box.innerHTML = `
        <div class="cb-title">${{title}}</div>
        <div class="cb-gradient-bar" style="background:${{grad}};"></div>
        <div class="cb-ticks"><span>${{minS}}</span><span>${{midS}}</span><span>${{maxS}}</span></div>
    `;
}}

function changeMapMode(mode) {{
    currentMapMode = mode;
    sa2GeoJsonLayer.setStyle(getSA2PolygonStyle);
    updateColorbar();
    refreshMapLabels();
}}

function fitCommuterCatchment() {{
    const targetLyr = sa2LayerByCode[selectedSA2Code];
    if (!targetLyr) return;
    let combinedBounds = targetLyr.getBounds();
    const {{ originMap }} = getActiveInflowMap();
    Object.keys(originMap).forEach(ocode => {{
        const olyr = sa2LayerByCode[ocode];
        if (olyr) combinedBounds.extend(olyr.getBounds());
    }});
    map.flyToBounds(combinedBounds, {{ padding: [36, 36], maxZoom: 12, duration: 0.75 }});
}}

function zoomToTargetSA2() {{
    const targetLyr = sa2LayerByCode[selectedSA2Code];
    if (targetLyr) {{
        map.flyToBounds(targetLyr.getBounds(), {{ padding: [45, 45], maxZoom: 13, duration: 0.75 }});
    }}
}}

function selectSA2(sa2Code, autoFit = true, updateDropdown = true) {{
    if (!sa2FeatureByCode[sa2Code]) return;
    selectedSA2Code = sa2Code;
    if (updateDropdown) {{
        const sel = document.getElementById('sa2-select');
        if (sel) sel.value = sa2Code;
    }}
    document.querySelectorAll('#quick-chips .area-chip').forEach(btn => {{
        btn.classList.toggle('active', btn.getAttribute('data-sa2') === sa2Code);
    }});

    sa2GeoJsonLayer.setStyle(getSA2PolygonStyle);
    updateReportSheet();
    updateColorbar();
    refreshMapLabels();

    if (autoFit) {{
        if (currentMapMode === 'commute') fitCommuterCatchment();
        else zoomToTargetSA2();
    }}
}}

function switchMobileView(viewName) {{
    const splitEl = document.getElementById('main-split');
    const btnReport = document.getElementById('btn-tab-report');
    const btnMap = document.getElementById('btn-tab-map');
    if (!splitEl || !btnReport || !btnMap) return;

    if (viewName === 'map') {{
        splitEl.classList.add('show-map');
        btnMap.classList.add('active');
        btnReport.classList.remove('active');
        setTimeout(() => {{
            map.invalidateSize();
            if (currentMapMode === 'commute') fitCommuterCatchment();
            else zoomToTargetSA2();
        }}, 160);
    }} else {{
        splitEl.classList.remove('show-map');
        btnReport.classList.add('active');
        btnMap.classList.remove('active');
    }}
}}

// Initial load
populateSA2Dropdown('');
selectSA2('202000', true, true);
</script>
</body>
</html>"""

# Write English & Japanese Dedicated Census Apps (census.html & census_ja.html)
census_html_en = build_census_html_app("en")
with open(OUTPUT_CENSUS_EN_PATH, "w", encoding="utf-8") as f:
    f.write(census_html_en)

census_html_ja = build_census_html_app("ja")
with open(OUTPUT_CENSUS_JA_PATH, "w", encoding="utf-8") as f:
    f.write(census_html_ja)

if DOCS_DIR.exists():
    shutil.copy2(OUTPUT_CENSUS_EN_PATH, DOCS_DIR / "census.html")
    shutil.copy2(OUTPUT_CENSUS_JA_PATH, DOCS_DIR / "census_ja.html")
    print(f"Synced census.html and census_ja.html to GitHub Pages directory: {DOCS_DIR.resolve()}")

cen_en_mb = OUTPUT_CENSUS_EN_PATH.stat().st_size / (1024 * 1024)
cen_ja_mb = OUTPUT_CENSUS_JA_PATH.stat().st_size / (1024 * 1024)
print(f"Generated Dedicated Census Report App (EN): {OUTPUT_CENSUS_EN_PATH.resolve()} ({cen_en_mb:.2f} MB)")
print(f"Generated Dedicated Census Report App (JA): {OUTPUT_CENSUS_JA_PATH.resolve()} ({cen_ja_mb:.2f} MB)")


Synced census.html and census_ja.html to GitHub Pages directory: C:\★大学院留学に向けて\01_On-site\11. Geothermal Research Project\★NEDO Project Phase 2\data\docs
Generated Dedicated Census Report App (EN): C:\★大学院留学に向けて\01_On-site\11. Geothermal Research Project\★NEDO Project Phase 2\data\map_app\census.html (1.80 MB)
Generated Dedicated Census Report App (JA): C:\★大学院留学に向けて\01_On-site\11. Geothermal Research Project\★NEDO Project Phase 2\data\map_app\census_ja.html (1.80 MB)
